# Audit read-only de préparation des données — Score d'attention dossier IRIS

> **Module :** Score d'attention dossier — Audit de préparation des données  
> **Projet :** IRIS Auto Fraud Decision Platform — BNA Assurances  
> **Statut :** Audit read-only — Aucune écriture en base  
> **Référence méthodologique :** `docs/scoring/claim_attention_scoring_methodology.md`  
> **Référence feature spec :** `docs/scoring/claim_attention_feature_specification.md`  
> **Auteur :** Wiem Benzarti

---

## Section 1 — Introduction et objectif

Ce notebook audite si la base PostgreSQL/DWH contient suffisamment de données fiables pour construire le futur **score d'attention dossier IRIS**.

### Questions auxquelles ce notebook doit répondre

| Question | Famille de signaux |
|---|---|
| Peut-on calculer la récurrence client ? | Récurrence client (P1) |
| Peut-on calculer la récurrence véhicule ? | Récurrence véhicule (P1) |
| Peut-on calculer les signaux de chronologie ? | Chronologie (P1) |
| Peut-on calculer l'atypicité du montant ? | Montant (P1) |
| Peut-on utiliser les indicateurs GEO ? | Cohérence géographique (P2) |
| Peut-on lier les dossiers au module VHS ? | VHS linkage (P2) |
| Peut-on utiliser les données tiers/conducteur ? | Récurrence tiers (P3) |
| Quelles familles sont READY / PARTIAL / NOT_READY / NOT_AVAILABLE ? | Matrice de synthèse |

### Positionnement métier

IRIS est une plateforme **d'aide à la décision**. Le score d'attention dossier ne constitue pas une preuve de fraude et n'accuse pas les clients. Il aide à prioriser l'analyse des dossiers sinistres et à identifier des **signaux à examiner**.

> **Les éléments produits par ce notebook servent uniquement à évaluer la préparation des données. Aucun score d'attention n'est calculé à ce stade.**

### Ce que ce notebook ne fait PAS

- ❌ Aucun calcul de score  
- ❌ Aucune implémentation de feature mart  
- ❌ Aucun modèle Machine Learning  
- ❌ Aucune écriture en base de données  
- ❌ Aucune modification des ETL existants  
- ❌ Aucune modification du module VHS

> Version canonique consolidee: structure read-only conservee, correctifs appliques pour cles techniques `0` et dates DWH `YYYYMMDD`. Le doublon `_CHAT` est retire pour eviter deux sources de verite.


---
## Section 2 — Garde-fou : exécution en lecture seule (Read-Only Safety Guard)

In [1]:
# =============================================================================
# SECTION 2 — READ-ONLY SAFETY GUARD
# All SQL execution in this notebook must go through safe_read_sql().
# Any query containing forbidden DML/DDL keywords will be rejected.
# =============================================================================

import re
import pandas as pd

# Forbidden SQL keywords — any of these will block execution
_FORBIDDEN_KEYWORDS = [
    r'\bINSERT\b',
    r'\bUPDATE\b',
    r'\bDELETE\b',
    r'\bDROP\b',
    r'\bALTER\b',
    r'\bTRUNCATE\b',
    r'\bMERGE\b',
    r'\bCREATE\b',
    r'\bGRANT\b',
    r'\bREVOKE\b',
]


def safe_read_sql(query: str, engine) -> pd.DataFrame:
    """
    Read-only SQL execution guard.

    Rejects any query containing forbidden DML/DDL keywords.
    Only SELECT and WITH (CTEs) queries are permitted.

    Parameters
    ----------
    query : str
        SQL query string to execute.
    engine : sqlalchemy.Engine
        SQLAlchemy engine connected to the DWH.

    Returns
    -------
    pd.DataFrame
        Query result as a DataFrame.

    Raises
    ------
    ValueError
        If a forbidden keyword is detected in the query.
    """
    query_upper = query.upper()
    for pattern in _FORBIDDEN_KEYWORDS:
        if re.search(pattern, query_upper):
            keyword = pattern.replace(r'\b', '').replace('\\b', '')
            raise ValueError(
                f"[SAFETY GUARD] Forbidden SQL keyword detected: '{keyword}'.\n"
                f"This notebook is READ-ONLY. No data modification is allowed."
            )

    stripped = query.strip().upper()
    if not (stripped.startswith('SELECT') or stripped.startswith('WITH')):
        raise ValueError(
            "[SAFETY GUARD] Only SELECT or WITH (CTE) queries are permitted in this notebook."
        )

    return pd.read_sql(query, engine)


print("[OK] Read-only safety guard loaded. All SQL must go through safe_read_sql().")

[OK] Read-only safety guard loaded. All SQL must go through safe_read_sql().


---
## Section 3 — Imports et configuration

In [2]:
# =============================================================================
# SECTION 3 — IMPORTS AND CONFIGURATION
# =============================================================================

import os
import sys
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Project root — resolves to IRIS_AUTO_FRAUD/
# Works whether the notebook is run from inside notebooks/validation_scoring/
# or from the project root.
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path().resolve()
if NOTEBOOK_DIR.name == 'validation_scoring':
    PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

# Ensure project root is on sys.path so etl.utils.runtime is importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"[INFO] Project root : {PROJECT_ROOT}")

# ---------------------------------------------------------------------------
# Output directory
# ---------------------------------------------------------------------------
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'quality_reports' / 'scoring' / 'data_readiness'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[INFO] Output directory : {OUTPUT_DIR}")

# ---------------------------------------------------------------------------
# Pandas display options
# ---------------------------------------------------------------------------
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.width', 200)

# ---------------------------------------------------------------------------
# Audit run metadata
# ---------------------------------------------------------------------------
RUN_TIMESTAMP = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
RUN_DATE = datetime.now().strftime('%Y-%m-%d')
print(f"[INFO] Audit run timestamp : {RUN_TIMESTAMP}")

# ---------------------------------------------------------------------------
# Global audit state — accumulated during notebook execution
# ---------------------------------------------------------------------------
AUDIT_STATE = {
    'db_connected': False,
    'best_claim_table': None,
    'best_claim_schema': None,
    'total_claim_rows': None,
    'client_key_col': None,
    'vehicle_key_col': None,
    'contract_key_col': None,
    'guarantee_key_col': None,
    'agency_key_col': None,
    'geo_key_col': None,
    'claim_date_col': None,
    'declaration_date_col': None,
    'amount_col': None,
    'claim_id_col': None,
}

# Readiness values used throughout the notebook
STATUS_READY        = 'READY'
STATUS_PARTIAL      = 'PARTIAL'
STATUS_NOT_READY    = 'NOT_READY'
STATUS_NOT_AVAILABLE = 'NOT_AVAILABLE'
STATUS_TO_AUDIT     = 'TO_AUDIT'

print("[OK] Imports and configuration complete.")

[INFO] Project root : C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD
[INFO] Output directory : C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\data_readiness
[INFO] Audit run timestamp : 2026-07-07 11:36:33
[OK] Imports and configuration complete.


---
## Section 4 — Connexion à la base de données

In [3]:
# =============================================================================
# SECTION 4 — DATABASE CONNECTION
# Uses the project's existing build_engine() utility from etl.utils.runtime.
# Credentials are read from .env (DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD).
# No credentials are printed.
# =============================================================================

engine = None
DB_CONNECTION_STATUS = 'FAILED'
DB_CONNECTION_NOTE = ''

try:
    from etl.utils.runtime import build_engine
    engine = build_engine()

    # Test connection with a harmless read-only query
    test_df = safe_read_sql('SELECT 1 AS connection_test', engine)
    assert test_df['connection_test'].iloc[0] == 1

    # Read DB_NAME safely for display (no password exposure)
    env_path = PROJECT_ROOT / '.env'
    _db_name = os.environ.get('DB_NAME', 'iris_auto_fraud')
    if env_path.exists():
        for line in env_path.read_text(encoding='utf-8', errors='replace').splitlines():
            if line.strip().startswith('DB_NAME='):
                _db_name = line.split('=', 1)[1].strip().strip('"\'')

    DB_CONNECTION_STATUS = 'OK'
    DB_CONNECTION_NOTE = f'Connected to database: {_db_name}'
    AUDIT_STATE['db_connected'] = True
    print(f"[OK] {DB_CONNECTION_NOTE}")
    print("[OK] Connection test (SELECT 1) passed.")

except ImportError as e:
    DB_CONNECTION_NOTE = f'Could not import etl.utils.runtime: {e}'
    print(f"[WARN] {DB_CONNECTION_NOTE}")
    print("[INFO] Attempting direct SQLAlchemy connection from .env variables...")

    try:
        from sqlalchemy import URL, create_engine as _create_engine

        def _get_env(key, default=''):
            env_path = PROJECT_ROOT / '.env'
            if env_path.exists():
                for line in env_path.read_text(encoding='utf-8', errors='replace').splitlines():
                    line = line.strip()
                    if line.startswith(f'{key}='):
                        return line.split('=', 1)[1].strip().strip('"\'')
            return os.environ.get(key, default)

        url = URL.create(
            drivername='postgresql+psycopg2',
            host=_get_env('DB_HOST', 'localhost'),
            port=int(_get_env('DB_PORT', '5432')),
            database=_get_env('DB_NAME', 'iris_auto_fraud'),
            username=_get_env('DB_USER', 'postgres'),
            password=_get_env('DB_PASSWORD', ''),
        )
        engine = _create_engine(url, pool_pre_ping=True)
        test_df = safe_read_sql('SELECT 1 AS connection_test', engine)
        assert test_df['connection_test'].iloc[0] == 1

        DB_CONNECTION_STATUS = 'OK'
        DB_CONNECTION_NOTE = f'Connected via direct SQLAlchemy URL'
        AUDIT_STATE['db_connected'] = True
        print(f"[OK] {DB_CONNECTION_NOTE}")
    except Exception as e2:
        DB_CONNECTION_NOTE = f'Connection failed: {e2}'
        print(f"[ERROR] {DB_CONNECTION_NOTE}")
        print("[INFO] Verify .env file contains DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD.")

except Exception as e:
    DB_CONNECTION_NOTE = f'Connection failed: {e}'
    print(f"[ERROR] {DB_CONNECTION_NOTE}")
    print("[INFO] Verify .env file and database availability.")

print(f"\nDB Connection status : {DB_CONNECTION_STATUS}")

[OK] Connected to database: iris_auto_fraud
[OK] Connection test (SELECT 1) passed.

DB Connection status : OK


---
## Section 5 — Inventaire des tables DWH (information_schema)

Inventaire des tables dans les schémas `staging`, `dwh`, `mart`, `public`.

In [4]:
# =============================================================================
# SECTION 5 — DWH SCHEMA/TABLE INVENTORY
# Read from information_schema — no assumptions about table names.
# =============================================================================

table_inventory_df = pd.DataFrame()

if not AUDIT_STATE['db_connected']:
    print("[SKIP] No database connection. Skipping table inventory.")
else:
    SCHEMAS_OF_INTEREST = ('staging', 'dwh', 'mart', 'public')
    schemas_list = "','".join(SCHEMAS_OF_INTEREST)

    inventory_query = f"""
    SELECT
        t.table_schema,
        t.table_name,
        t.table_type,
        COALESCE(s.n_live_tup, -1) AS row_count_estimate
    FROM information_schema.tables t
    LEFT JOIN pg_stat_user_tables s
        ON s.schemaname = t.table_schema
        AND s.relname = t.table_name
    WHERE t.table_schema IN ('{schemas_list}')
    ORDER BY t.table_schema, t.table_name
    """

    try:
        table_inventory_df = safe_read_sql(inventory_query, engine)
        print(f"[OK] Found {len(table_inventory_df)} tables/views across schemas: {SCHEMAS_OF_INTEREST}")

        # Infer candidate domain from table name
        DOMAIN_KEYWORDS = {
            'claims': ['sinistre', 'claim', 'snt', 'sini'],
            'client': ['client', 'assure', 'pers'],
            'contract': ['contrat', 'contract', 'production', 'police', 'avenant'],
            'vehicle': ['vehicule', 'vehicle', 'vehic', 'immat', 'engin'],
            'guarantee': ['garantie', 'guarantee', 'grnt'],
            'agency': ['agence', 'agency', 'agenc'],
            'geo': ['geo', 'region', 'gouvern', 'deleg', 'localit', 'adresse'],
            'third_party': ['tiers', 'third', 'adversaire', 'advers'],
            'driver': ['conducteur', 'driver', 'chauff'],
            'vhs': ['vhs', 'inspection', 'staffim', 'technical'],
            'reference': ['ref', 'mapping', 'code', 'nomenclature', 'dim_'],
        }

        def infer_domain(name: str) -> str:
            name_lower = name.lower()
            for domain, keywords in DOMAIN_KEYWORDS.items():
                if any(kw in name_lower for kw in keywords):
                    return domain
            return 'unknown'

        table_inventory_df['candidate_domain'] = table_inventory_df['table_name'].apply(infer_domain)

        print("\nTable inventory by schema:")
        print(table_inventory_df.groupby(['table_schema', 'candidate_domain']).size().reset_index(name='count').to_string(index=False))

        # Export
        out_path = OUTPUT_DIR / 'claim_scoring_table_inventory.csv'
        table_inventory_df.to_csv(out_path, index=False, encoding='utf-8-sig')
        print(f"\n[EXPORT] Table inventory saved: {out_path}")

    except Exception as e:
        print(f"[ERROR] Could not retrieve table inventory: {e}")

table_inventory_df.head(20)

[OK] Found 23 tables/views across schemas: ('staging', 'dwh', 'mart', 'public')

Table inventory by schema:
table_schema candidate_domain  count
         dwh           claims      2
         dwh           client      1
         dwh         contract      2
         dwh           driver      1
         dwh              geo      1
         dwh        guarantee      1
         dwh        reference      4
         dwh      third_party      1
         dwh          vehicle      2
         dwh              vhs      1
        mart        reference      1
        mart              vhs      2
     staging           claims      1
     staging           client      1
     staging         contract      1
     staging              vhs      1

[EXPORT] Table inventory saved: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\data_readiness\claim_scoring_table_inventory.csv


,table_schema,table_name,table_type,row_count_estimate,candidate_domain
0,dwh,dim_camtier,BASE TABLE,23,reference
1,dwh,dim_client,BASE TABLE,461881,client
2,dwh,dim_conducteur,BASE TABLE,161523,driver
3,dwh,dim_contrat,BASE TABLE,447585,contract
4,dwh,dim_date,BASE TABLE,9496,reference
5,dwh,dim_garantie,BASE TABLE,334,guarantee
6,dwh,dim_geo,BASE TABLE,1856,geo
7,dwh,dim_intermediaire,BASE TABLE,212,reference
8,dwh,dim_produit,BASE TABLE,37,reference
9,dwh,dim_sinistre,BASE TABLE,231496,claims


---
## Section 6 — Découverte de la table centrale des sinistres

Recherche des tables candidates contenant les dossiers sinistres automobiles.

In [5]:
# =============================================================================
# SECTION 6 — CLAIM CENTRAL TABLE DISCOVERY
# Search table names for claim-related keywords, then inspect their columns.
# =============================================================================

claim_table_candidates_df = pd.DataFrame()

if not AUDIT_STATE['db_connected'] or table_inventory_df.empty:
    print("[SKIP] No database connection or no table inventory available.")
else:
    CLAIM_KEYWORDS = ['sinistre', 'claim', 'snt', 'sini', 'fact_sin', 'fact_clm']

    # Candidate column keyword groups
    COL_GROUPS = {
        'key': ['claim_sk', 'sinistre_sk', 'numsnt', 'num_sinistre', 'numsini', 'id_sinistre', 'snt_sk'],
        'guarantee': ['garantie', 'grntsini', 'guarantee', 'grnt', 'nature_sinistre'],
        'client': ['client_sk', 'cnat', 'numpers', 'id_client', 'assure_sk'],
        'contract': ['contract_sk', 'numcnt', 'id_contrat', 'contrat_sk', 'police'],
        'vehicle': ['vehicle_sk', 'vehicule_sk', 'immatriculation', 'matricule', 'id_vehicule'],
        'agency': ['agence', 'agency_sk', 'agence_sk', 'id_agence'],
        'geo': ['geo_sk', 'region', 'gouvernorat', 'localite', 'lieu', 'delegation', 'adresse'],
        'date_claim': ['date_sinistre', 'claim_date', 'dtsurv', 'date_survenance', 'dt_survenance'],
        'date_declaration': ['date_declaration', 'declaration_date', 'dt_declaration', 'date_decl'],
        'amount': ['montant', 'amount', 'cout', 'indemnisation', 'evaluation', 'capital', 'prestation'],
    }

    # Filter candidate tables
    candidate_mask = table_inventory_df['table_name'].str.lower().apply(
        lambda n: any(kw in n for kw in CLAIM_KEYWORDS)
    )
    candidate_tables = table_inventory_df[candidate_mask].copy()
    print(f"[INFO] Found {len(candidate_tables)} candidate claim table(s):")
    print(candidate_tables[['table_schema', 'table_name', 'row_count_estimate']].to_string(index=False))

    records = []
    for _, row in candidate_tables.iterrows():
        schema = row['table_schema']
        table = row['table_name']
        fqtn = f'"{schema}"."{table}"'

        # Get columns from information_schema
        col_query = f"""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = '{schema}' AND table_name = '{table}'
        ORDER BY ordinal_position
        """
        try:
            col_df = safe_read_sql(col_query, engine)
            all_cols = col_df['column_name'].str.lower().tolist()
        except Exception as e:
            print(f"  [WARN] Could not get columns for {fqtn}: {e}")
            continue

        detected = {grp: [] for grp in COL_GROUPS}
        score = 0
        for grp, keywords in COL_GROUPS.items():
            for kw in keywords:
                matches = [c for c in all_cols if kw.lower() in c]
                detected[grp].extend(matches)
            if detected[grp]:
                score += 1

        def first_or_none(lst):
            return lst[0] if lst else None

        records.append({
            'table_schema': schema,
            'table_name': table,
            'row_count_estimate': row.get('row_count_estimate', -1),
            'candidate_score': score,
            'detected_key_columns': ', '.join(set(detected['key'])) or 'NONE',
            'detected_date_columns': ', '.join(set(detected['date_claim'] + detected['date_declaration'])) or 'NONE',
            'detected_amount_columns': ', '.join(set(detected['amount'])) or 'NONE',
            'detected_link_columns': ', '.join(set(detected['client'] + detected['vehicle'] + detected['contract'] + detected['agency'])) or 'NONE',
            'detected_geo_columns': ', '.join(set(detected['geo'])) or 'NONE',
            '_client_col': first_or_none(detected['client']),
            '_vehicle_col': first_or_none(detected['vehicle']),
            '_contract_col': first_or_none(detected['contract']),
            '_guarantee_col': first_or_none(detected['guarantee']),
            '_agency_col': first_or_none(detected['agency']),
            '_geo_col': first_or_none(detected['geo']),
            '_claim_date_col': first_or_none(detected['date_claim']),
            '_declaration_date_col': first_or_none(detected['date_declaration']),
            '_amount_col': first_or_none(detected['amount']),
            '_claim_id_col': first_or_none(detected['key']),
            'recommendation': 'BEST_CANDIDATE' if score >= 5 else ('CANDIDATE' if score >= 3 else 'WEAK_CANDIDATE'),
        })

    if records:
        claim_table_candidates_df = pd.DataFrame(records).sort_values('candidate_score', ascending=False).reset_index(drop=True)

        # Select best candidate
        best = claim_table_candidates_df.iloc[0]
        AUDIT_STATE['best_claim_schema'] = best['table_schema']
        AUDIT_STATE['best_claim_table'] = best['table_name']
        AUDIT_STATE['client_key_col'] = best['_client_col']
        AUDIT_STATE['vehicle_key_col'] = best['_vehicle_col']
        AUDIT_STATE['contract_key_col'] = best['_contract_col']
        AUDIT_STATE['guarantee_key_col'] = best['_guarantee_col']
        AUDIT_STATE['agency_key_col'] = best['_agency_col']
        AUDIT_STATE['geo_key_col'] = best['_geo_col']
        AUDIT_STATE['claim_date_col'] = best['_claim_date_col']
        AUDIT_STATE['declaration_date_col'] = best['_declaration_date_col']
        AUDIT_STATE['amount_col'] = best['_amount_col']
        AUDIT_STATE['claim_id_col'] = best['_claim_id_col']

        print(f"\n[DECISION] Best claim table candidate: {best['table_schema']}.{best['table_name']}")
        print(f"  Candidate score: {best['candidate_score']}/10")
        print(f"  Key columns   : {best['detected_key_columns']}")
        print(f"  Date columns  : {best['detected_date_columns']}")
        print(f"  Amount columns: {best['detected_amount_columns']}")
        print(f"  Link columns  : {best['detected_link_columns']}")

        # Export (remove internal _ columns)
        export_cols = [c for c in claim_table_candidates_df.columns if not c.startswith('_')]
        out_path = OUTPUT_DIR / 'claim_scoring_claim_table_candidates.csv'
        claim_table_candidates_df[export_cols].to_csv(out_path, index=False, encoding='utf-8-sig')
        print(f"\n[EXPORT] Claim table candidates saved: {out_path}")
    else:
        print("[WARN] No claim table candidates found. Check schema/table naming conventions.")

if not claim_table_candidates_df.empty:
    export_cols = [c for c in claim_table_candidates_df.columns if not c.startswith('_')]
    display(claim_table_candidates_df[export_cols])

[INFO] Found 3 candidate claim table(s):
table_schema    table_name  row_count_estimate
         dwh  dim_sinistre              231496
         dwh fact_sinistre              381893
     staging stg_sinistres              381863

[DECISION] Best claim table candidate: dwh.fact_sinistre
  Candidate score: 8/10
  Key columns   : fact_sinistre_sk, sinistre_sk, geo_sinistre_sk
  Date columns  : date_survenance_sk, date_declaration_sk
  Amount columns: montant_reserve, montant_recours, montant_evaluation, montant_charge_sinistre, montant_reglement
  Link columns  : contrat_sk, vehicule_sk, client_sk

[EXPORT] Claim table candidates saved: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\data_readiness\claim_scoring_claim_table_candidates.csv


,table_schema,table_name,row_count_estimate,candidate_score,detected_key_columns,detected_date_columns,detected_amount_columns,detected_link_columns,detected_geo_columns,recommendation
0,dwh,fact_sinistre,381893,8,"fact_sinistre_sk, sinistre_sk, geo_sinistre_sk","date_survenance_sk, date_declaration_sk","montant_reserve, montant_recours, montant_eval...","contrat_sk, vehicule_sk, client_sk",NONE,BEST_CANDIDATE
1,staging,stg_sinistres,381863,4,"numsnttie, numsnt_norm, numsnt_1, numsnt",dtsurv,NONE,"numcnt_norm, numcnt, numcnttie",NONE,CANDIDATE
2,dwh,dim_sinistre,231496,1,sinistre_sk,NONE,NONE,NONE,NONE,WEAK_CANDIDATE


---
## Section 7 — Audit de couverture des clés

> **Note métier :** Les clés manquantes réduisent la confiance de l'analyse. Elles constituent des **limitations de qualité des données**, et non des signaux de suspicion.

In [6]:
# =============================================================================
# SECTION 7 — KEY COVERAGE AUDIT
# Compute coverage percentage for all critical join keys.
# If a column does not exist, it is marked NOT_AVAILABLE without failing.
# =============================================================================

key_coverage_records = []

if not AUDIT_STATE['db_connected'] or not AUDIT_STATE['best_claim_table']:
    print("[SKIP] No claim table identified. Skipping key coverage audit.")
else:
    schema = AUDIT_STATE['best_claim_schema']
    table = AUDIT_STATE['best_claim_table']
    fqtn = f'"{schema}"."{table}"'

    print(f"[INFO] Running key coverage audit on: {fqtn}")

    # Get total row count
    try:
        total_df = safe_read_sql(f'SELECT COUNT(*) AS total_rows FROM {fqtn}', engine)
        total_rows = int(total_df['total_rows'].iloc[0])
        AUDIT_STATE['total_claim_rows'] = total_rows
        print(f"  Total rows: {total_rows:,}")
    except Exception as e:
        print(f"  [ERROR] Could not count rows: {e}")
        total_rows = 0

    # Get all column names in the table
    try:
        col_info_df = safe_read_sql(
            f"SELECT column_name FROM information_schema.columns WHERE table_schema='{schema}' AND table_name='{table}'",
            engine
        )
        existing_cols = col_info_df['column_name'].str.lower().tolist()
    except:
        existing_cols = []

    # Key definitions: (display_name, audit_state_key, candidate_column_names)
    KEY_DEFINITIONS = [
        ('claim_id', 'claim_id_col', ['claim_sk', 'sinistre_sk', 'numsnt', 'numsini', 'id_sinistre', 'snt_sk']),
        ('client_key', 'client_key_col', ['client_sk', 'cnat', 'numpers', 'id_client', 'assure_sk']),
        ('vehicle_key', 'vehicle_key_col', ['vehicle_sk', 'vehicule_sk', 'immatriculation', 'matricule']),
        ('contract_key', 'contract_key_col', ['contract_sk', 'numcnt', 'id_contrat', 'contrat_sk']),
        ('guarantee_key', 'guarantee_key_col', ['garantie', 'grntsini', 'guarantee', 'grnt', 'nature_sinistre']),
        ('agency_key', 'agency_key_col', ['agence', 'agency_sk', 'agence_sk', 'id_agence']),
        ('geo_key', 'geo_key_col', ['geo_sk', 'region', 'gouvernorat', 'localite', 'lieu', 'delegation']),
        ('claim_date', 'claim_date_col', ['date_sinistre', 'claim_date', 'dtsurv', 'date_survenance']),
        ('declaration_date', 'declaration_date_col', ['date_declaration', 'declaration_date', 'dt_declaration']),
        ('amount', 'amount_col', ['montant', 'amount', 'cout', 'indemnisation', 'evaluation', 'prestation']),
    ]

    for display_name, state_key, candidates in KEY_DEFINITIONS:
        # Find first matching column
        found_col = AUDIT_STATE.get(state_key)
        if not found_col:
            found_col = next((c for c in candidates if c in existing_cols), None)
        if found_col and found_col not in existing_cols:
            # Try fuzzy: any col containing the candidate
            fuzzy = [c for c in existing_cols if found_col in c]
            found_col = fuzzy[0] if fuzzy else None

        if not found_col:
            # Try fuzzy match for any candidate
            for cand in candidates:
                fuzzy = [c for c in existing_cols if cand in c]
                if fuzzy:
                    found_col = fuzzy[0]
                    break

        if not found_col:
            key_coverage_records.append({
                'key_name': display_name,
                'column_found': 'NOT_AVAILABLE',
                'non_null_count': 'N/A',
                'null_count': 'N/A',
                'coverage_pct': 'N/A',
                'status': STATUS_NOT_AVAILABLE,
                'note': 'Column not detected in table',
            })
            print(f"  [{display_name}] → NOT_AVAILABLE (no matching column found)")
            continue

        # Update audit state with confirmed column
        if state_key in AUDIT_STATE:
            AUDIT_STATE[state_key] = found_col

        try:
            cov_df = safe_read_sql(
                f'SELECT COUNT("{found_col}") AS non_null_count, COUNT(*) - COUNT("{found_col}") AS null_count FROM {fqtn}',
                engine
            )
            non_null = int(cov_df['non_null_count'].iloc[0])
            null_cnt = int(cov_df['null_count'].iloc[0])
            pct = round(non_null / total_rows * 100, 2) if total_rows > 0 else 0.0

            if pct >= 90:
                status = STATUS_READY
            elif pct >= 60:
                status = STATUS_PARTIAL
            elif pct > 0:
                status = STATUS_NOT_READY
            else:
                status = STATUS_NOT_AVAILABLE

            key_coverage_records.append({
                'key_name': display_name,
                'column_found': found_col,
                'non_null_count': non_null,
                'null_count': null_cnt,
                'coverage_pct': pct,
                'status': status,
                'note': f'Coverage {pct}%',
            })
            print(f"  [{display_name}] col='{found_col}' → {non_null:,} non-null / {total_rows:,} total ({pct}%) — {status}")
        except Exception as e:
            key_coverage_records.append({
                'key_name': display_name,
                'column_found': found_col,
                'non_null_count': 'ERROR',
                'null_count': 'ERROR',
                'coverage_pct': 'ERROR',
                'status': STATUS_TO_AUDIT,
                'note': str(e),
            })
            print(f"  [{display_name}] ERROR: {e}")

    key_coverage_df = pd.DataFrame(key_coverage_records)
    out_path = OUTPUT_DIR / 'claim_scoring_key_coverage.csv'
    key_coverage_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"\n[EXPORT] Key coverage saved: {out_path}")
    display(key_coverage_df)

[INFO] Running key coverage audit on: "dwh"."fact_sinistre"
  Total rows: 381,893
  [claim_id] col='fact_sinistre_sk' → 381,893 non-null / 381,893 total (100.0%) — READY
  [client_key] col='client_sk' → 381,893 non-null / 381,893 total (100.0%) — READY
  [vehicle_key] col='vehicule_sk' → 381,893 non-null / 381,893 total (100.0%) — READY
  [contract_key] col='contrat_sk' → 381,893 non-null / 381,893 total (100.0%) — READY
  [guarantee_key] col='code_garantie' → 381,893 non-null / 381,893 total (100.0%) — READY
  [agency_key] → NOT_AVAILABLE (no matching column found)
  [geo_key] → NOT_AVAILABLE (no matching column found)
  [claim_date] col='date_survenance_sk' → 381,893 non-null / 381,893 total (100.0%) — READY
  [declaration_date] col='date_declaration_sk' → 381,893 non-null / 381,893 total (100.0%) — READY
  [amount] col='montant_evaluation' → 381,893 non-null / 381,893 total (100.0%) — READY

[EXPORT] Key coverage saved: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality

,key_name,column_found,non_null_count,null_count,coverage_pct,status,note
0,claim_id,fact_sinistre_sk,381893,0,100.0000,READY,Coverage 100.0%
1,client_key,client_sk,381893,0,100.0000,READY,Coverage 100.0%
2,vehicle_key,vehicule_sk,381893,0,100.0000,READY,Coverage 100.0%
3,contract_key,contrat_sk,381893,0,100.0000,READY,Coverage 100.0%
4,guarantee_key,code_garantie,381893,0,100.0000,READY,Coverage 100.0%
5,agency_key,NOT_AVAILABLE,N/A,N/A,N/A,NOT_AVAILABLE,Column not detected in table
6,geo_key,NOT_AVAILABLE,N/A,N/A,N/A,NOT_AVAILABLE,Column not detected in table
7,claim_date,date_survenance_sk,381893,0,100.0000,READY,Coverage 100.0%
8,declaration_date,date_declaration_sk,381893,0,100.0000,READY,Coverage 100.0%
9,amount,montant_evaluation,381893,0,100.0000,READY,Coverage 100.0%


---
## Section 8 — Audit de qualité des dates

In [7]:
# =============================================================================
# SECTION 8 — DATE QUALITY AUDIT
# =============================================================================

date_quality_records = []

if not AUDIT_STATE['db_connected'] or not AUDIT_STATE['best_claim_table']:
    print("[SKIP] No claim table. Skipping date quality audit.")
else:
    schema = AUDIT_STATE['best_claim_schema']
    table = AUDIT_STATE['best_claim_table']
    fqtn = f'"{schema}"."{table}"'
    claim_date_col = AUDIT_STATE.get('claim_date_col')
    decl_date_col = AUDIT_STATE.get('declaration_date_col')
    total_rows = AUDIT_STATE.get('total_claim_rows', 0)

    print(f"[INFO] Date quality audit on: {fqtn}")
    print(f"  claim_date col      : {claim_date_col}")
    print(f"  declaration_date col: {decl_date_col}")

    # --- Claim date stats
    if claim_date_col:
        try:
            q = f"""
            SELECT
                COUNT(*) AS total,
                COUNT("{claim_date_col}") AS non_null,
                COUNT(*) - COUNT("{claim_date_col}") AS null_count,
                MIN("{claim_date_col}") AS min_date,
                MAX("{claim_date_col}") AS max_date,
                SUM(CASE WHEN "{claim_date_col}" > 0 AND "{claim_date_col}" < 20190101 THEN 1 ELSE 0 END) AS before_2019,
                SUM(CASE WHEN "{claim_date_col}" >= 20190101 THEN 1 ELSE 0 END) AS from_2019_onward
            FROM {fqtn}
            """
            df = safe_read_sql(q, engine)
            r = df.iloc[0]
            date_quality_records.append({'metric': 'claim_date_non_null', 'value': int(r['non_null']), 'note': f'{round(int(r["non_null"])/total_rows*100,2) if total_rows>0 else 0}%'})
            date_quality_records.append({'metric': 'claim_date_null_count', 'value': int(r['null_count']), 'note': ''})
            date_quality_records.append({'metric': 'claim_date_min', 'value': str(r['min_date']), 'note': ''})
            date_quality_records.append({'metric': 'claim_date_max', 'value': str(r['max_date']), 'note': ''})
            date_quality_records.append({'metric': 'claims_before_2019', 'value': int(r['before_2019']), 'note': 'migration_2019_flag candidates'})
            date_quality_records.append({'metric': 'claims_from_2019_onward', 'value': int(r['from_2019_onward']), 'note': ''})
            print(f"  claim_date: {int(r['non_null']):,} non-null | min={r['min_date']} | max={r['max_date']} | before_2019={int(r['before_2019']):,}")
        except Exception as e:
            date_quality_records.append({'metric': 'claim_date_audit', 'value': 'ERROR', 'note': str(e)})
            print(f"  [ERROR] claim_date audit: {e}")
    else:
        date_quality_records.append({'metric': 'claim_date_audit', 'value': 'NOT_AVAILABLE', 'note': 'No claim date column found'})

    # --- Declaration date stats + days_claim_to_declaration
    if claim_date_col and decl_date_col:
        try:
            q = f"""
            SELECT
                COUNT("{decl_date_col}") AS decl_non_null,
                COUNT(*) - COUNT("{decl_date_col}") AS decl_null,
                MIN("{decl_date_col}") AS decl_min,
                MAX("{decl_date_col}") AS decl_max,
                AVG((TO_DATE(NULLIF("{decl_date_col}", 0)::TEXT, 'YYYYMMDD') - TO_DATE(NULLIF("{claim_date_col}", 0)::TEXT, 'YYYYMMDD'))) AS avg_days_to_decl,
                MIN((TO_DATE(NULLIF("{decl_date_col}", 0)::TEXT, 'YYYYMMDD') - TO_DATE(NULLIF("{claim_date_col}", 0)::TEXT, 'YYYYMMDD'))) AS min_days_to_decl,
                MAX((TO_DATE(NULLIF("{decl_date_col}", 0)::TEXT, 'YYYYMMDD') - TO_DATE(NULLIF("{claim_date_col}", 0)::TEXT, 'YYYYMMDD'))) AS max_days_to_decl,
                SUM(CASE WHEN (TO_DATE(NULLIF("{decl_date_col}", 0)::TEXT, 'YYYYMMDD') - TO_DATE(NULLIF("{claim_date_col}", 0)::TEXT, 'YYYYMMDD')) < 0 THEN 1 ELSE 0 END) AS negative_decl_days
            FROM {fqtn}
            WHERE "{claim_date_col}" IS NOT NULL AND "{claim_date_col}" <> 0 AND "{decl_date_col}" IS NOT NULL AND "{decl_date_col}" <> 0
            """
            df = safe_read_sql(q, engine)
            r = df.iloc[0]
            date_quality_records.append({'metric': 'declaration_date_non_null', 'value': int(r['decl_non_null']), 'note': ''})
            date_quality_records.append({'metric': 'declaration_date_null', 'value': int(r['decl_null']), 'note': ''})
            date_quality_records.append({'metric': 'declaration_date_min', 'value': str(r['decl_min']), 'note': ''})
            date_quality_records.append({'metric': 'declaration_date_max', 'value': str(r['decl_max']), 'note': ''})
            date_quality_records.append({'metric': 'avg_days_claim_to_declaration', 'value': round(float(r['avg_days_to_decl'] or 0), 1), 'note': ''})
            date_quality_records.append({'metric': 'min_days_claim_to_declaration', 'value': int(r['min_days_to_decl'] or 0), 'note': ''})
            date_quality_records.append({'metric': 'max_days_claim_to_declaration', 'value': int(r['max_days_to_decl'] or 0), 'note': ''})
            date_quality_records.append({'metric': 'negative_days_claim_to_declaration', 'value': int(r['negative_decl_days'] or 0), 'note': 'Data quality anomaly — date inversion'})
            print(f"  days_claim_to_declaration: avg={r['avg_days_to_decl']:.1f} | min={r['min_days_to_decl']} | max={r['max_days_to_decl']} | negatives={int(r['negative_decl_days'] or 0):,}")
        except Exception as e:
            date_quality_records.append({'metric': 'days_claim_to_declaration', 'value': 'ERROR', 'note': str(e)})
            print(f"  [ERROR] days_claim_to_declaration: {e}")
    elif decl_date_col:
        date_quality_records.append({'metric': 'declaration_date', 'value': 'PARTIAL', 'note': 'Found declaration_date but no claim_date for diff'})
    else:
        date_quality_records.append({'metric': 'declaration_date_audit', 'value': 'NOT_AVAILABLE', 'note': ''})

    date_quality_df = pd.DataFrame(date_quality_records)
    out_path = OUTPUT_DIR / 'claim_scoring_date_quality.csv'
    date_quality_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"\n[EXPORT] Date quality saved: {out_path}")
    display(date_quality_df)

[INFO] Date quality audit on: "dwh"."fact_sinistre"
  claim_date col      : date_survenance_sk
  declaration_date col: date_declaration_sk
  [ERROR] claim_date audit: Execution failed on sql '
            SELECT
                COUNT(*) AS total,
                COUNT("date_survenance_sk") AS non_null,
                COUNT(*) - COUNT("date_survenance_sk") AS null_count,
                MIN("date_survenance_sk") AS min_date,
                MAX("date_survenance_sk") AS max_date,
                SUM(CASE WHEN "date_survenance_sk" < '2019-01-01' THEN 1 ELSE 0 END) AS before_2019,
                SUM(CASE WHEN "date_survenance_sk" >= '2019-01-01' THEN 1 ELSE 0 END) AS from_2019_onward
            FROM "dwh"."fact_sinistre"
            ': (psycopg2.errors.InvalidTextRepresentation) ERREUR:  syntaxe en entrée invalide pour le type bigint : « 2019-01-01 »
LINE 8: ...             SUM(CASE WHEN "date_survenance_sk" < '2019-01-0...
                                                             ^


,metric,value,note
0,claim_date_audit,ERROR,Execution failed on sql '\n SELECT\...
1,declaration_date_non_null,381893,
2,declaration_date_null,0,
3,declaration_date_min,0.0,
4,declaration_date_max,20261202.0,
5,avg_days_claim_to_declaration,461.4000,
6,min_days_claim_to_declaration,-20251020,
7,max_days_claim_to_declaration,79783,
8,negative_days_claim_to_declaration,51118,Data quality anomaly — date inversion


---
## Section 9 — Audit de qualité des montants

In [8]:
# =============================================================================
# SECTION 9 — AMOUNT QUALITY AUDIT
# =============================================================================

amount_quality_records = []

if not AUDIT_STATE['db_connected'] or not AUDIT_STATE['best_claim_table']:
    print("[SKIP] No claim table. Skipping amount quality audit.")
else:
    schema = AUDIT_STATE['best_claim_schema']
    table = AUDIT_STATE['best_claim_table']
    fqtn = f'"{schema}"."{table}"'
    amount_col = AUDIT_STATE.get('amount_col')
    total_rows = AUDIT_STATE.get('total_claim_rows', 0)

    print(f"[INFO] Amount quality audit on: {fqtn}")
    print(f"  Amount column: {amount_col}")

    if not amount_col:
        amount_quality_records.append({'metric': 'amount_col', 'value': 'NOT_AVAILABLE', 'note': 'No amount column detected'})
        print("  [WARN] No amount column found.")
    else:
        try:
            q = f"""
            SELECT
                COUNT(*) AS total_rows,
                COUNT("{amount_col}") AS non_null_count,
                COUNT(*) - COUNT("{amount_col}") AS null_count,
                SUM(CASE WHEN "{amount_col}" = 0 THEN 1 ELSE 0 END) AS zero_count,
                SUM(CASE WHEN "{amount_col}" < 0 THEN 1 ELSE 0 END) AS negative_count,
                MIN("{amount_col}") AS min_amount,
                MAX("{amount_col}") AS max_amount,
                AVG("{amount_col}") AS mean_amount,
                PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY "{amount_col}") AS median_amount,
                PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY "{amount_col}") AS p75_amount,
                PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY "{amount_col}") AS p90_amount,
                PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY "{amount_col}") AS p95_amount,
                PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY "{amount_col}") AS p99_amount
            FROM {fqtn}
            """
            df = safe_read_sql(q, engine)
            r = df.iloc[0]

            metrics = [
                ('total_rows', int(r['total_rows']), ''),
                ('non_null_count', int(r['non_null_count']), f"{round(int(r['non_null_count'])/total_rows*100,2) if total_rows>0 else 0}%"),
                ('null_count', int(r['null_count']), ''),
                ('zero_count', int(r['zero_count']), ''),
                ('negative_count', int(r['negative_count']), 'Data quality anomaly'),
                ('min_amount', round(float(r['min_amount'] or 0), 2), ''),
                ('max_amount', round(float(r['max_amount'] or 0), 2), ''),
                ('mean_amount', round(float(r['mean_amount'] or 0), 2), ''),
                ('median_amount', round(float(r['median_amount'] or 0), 2), ''),
                ('p75_amount', round(float(r['p75_amount'] or 0), 2), ''),
                ('p90_amount', round(float(r['p90_amount'] or 0), 2), ''),
                ('p95_amount', round(float(r['p95_amount'] or 0), 2), ''),
                ('p99_amount', round(float(r['p99_amount'] or 0), 2), ''),
            ]
            for name, val, note in metrics:
                amount_quality_records.append({'metric': name, 'value': val, 'note': note})

            print(f"  non_null={int(r['non_null_count']):,} | null={int(r['null_count']):,} | median={r['median_amount']:.2f} | p99={r['p99_amount']:.2f}")
        except Exception as e:
            amount_quality_records.append({'metric': 'amount_stats', 'value': 'ERROR', 'note': str(e)})
            print(f"  [ERROR] Amount stats: {e}")

        # Amount by guarantee if guarantee column exists
        grnt_col = AUDIT_STATE.get('guarantee_key_col')
        if grnt_col and amount_col:
            try:
                q_grnt = f"""
                SELECT
                    "{grnt_col}" AS guarantee,
                    COUNT(*) AS claim_count,
                    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY "{amount_col}") AS median_amount,
                    PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY "{amount_col}") AS p90_amount
                FROM {fqtn}
                WHERE "{amount_col}" IS NOT NULL
                GROUP BY "{grnt_col}"
                ORDER BY claim_count DESC
                LIMIT 20
                """
                grnt_df = safe_read_sql(q_grnt, engine)
                out_path = OUTPUT_DIR / 'claim_scoring_amount_by_guarantee.csv'
                grnt_df.to_csv(out_path, index=False, encoding='utf-8-sig')
                print(f"  Amount by guarantee: {len(grnt_df)} guarantee groups — saved to {out_path.name}")
            except Exception as e:
                print(f"  [WARN] Could not compute amount by guarantee: {e}")

    amount_quality_df = pd.DataFrame(amount_quality_records)
    out_path = OUTPUT_DIR / 'claim_scoring_amount_quality.csv'
    amount_quality_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"\n[EXPORT] Amount quality saved: {out_path}")
    display(amount_quality_df)

[INFO] Amount quality audit on: "dwh"."fact_sinistre"
  Amount column: montant_evaluation
  non_null=381,893 | null=0 | median=550.00 | p99=26700.00
  Amount by guarantee: 20 guarantee groups — saved to claim_scoring_amount_by_guarantee.csv

[EXPORT] Amount quality saved: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\data_readiness\claim_scoring_amount_quality.csv


,metric,value,note
0,total_rows,381893.0000,
1,non_null_count,381893.0000,100.0%
2,null_count,0.0000,
3,zero_count,36610.0000,
4,negative_count,0.0000,Data quality anomaly
5,min_amount,0.0000,
6,max_amount,40706064.0000,
7,mean_amount,2263.9200,
8,median_amount,550.0000,
9,p75_amount,2084.9500,


---
## Section 10 — Préparation de la récurrence client

Évaluation de la capacité à calculer les features de récurrence client (`client_claim_count_total`, `client_claim_count_12m`, `client_claim_count_24m`, `days_since_previous_claim`, `client_claim_frequency_band`).

In [9]:
# =============================================================================
# SECTION 10 — CLIENT RECURRENCE READINESS
# =============================================================================

client_recurrence_records = []
CLIENT_RECURRENCE_STATUS = STATUS_TO_AUDIT

if not AUDIT_STATE['db_connected'] or not AUDIT_STATE['best_claim_table']:
    print("[SKIP] No claim table. Skipping client recurrence readiness.")
else:
    schema = AUDIT_STATE['best_claim_schema']
    table = AUDIT_STATE['best_claim_table']
    fqtn = f'"{schema}"."{table}"'
    client_col = AUDIT_STATE.get('client_key_col')
    claim_date_col = AUDIT_STATE.get('claim_date_col')
    total_rows = AUDIT_STATE.get('total_claim_rows', 0)

    print(f"[INFO] Client recurrence readiness — client_col='{client_col}', claim_date_col='{claim_date_col}'")

    if not client_col:
        CLIENT_RECURRENCE_STATUS = STATUS_NOT_AVAILABLE
        client_recurrence_records.append({'feature': 'client_recurrence', 'status': STATUS_NOT_AVAILABLE, 'note': 'No client key column found'})
        print("  [WARN] Client key column not found → NOT_AVAILABLE")
    elif not claim_date_col:
        CLIENT_RECURRENCE_STATUS = STATUS_NOT_READY
        client_recurrence_records.append({'feature': 'client_recurrence', 'status': STATUS_NOT_READY, 'note': 'Client key found but claim_date missing'})
        print("  [WARN] claim_date missing → NOT_READY")
    else:
        try:
            q = f"""
            SELECT
                COUNT(DISTINCT NULLIF("{client_col}", 0)) AS distinct_clients,
                COUNT(*) AS total_claims,
                SUM(CASE WHEN "{client_col}" IS NULL OR "{client_col}" = 0 THEN 1 ELSE 0 END) AS missing_client,
                SUM(CASE WHEN "{client_col}" IS NOT NULL AND "{client_col}" <> 0 THEN 1 ELSE 0 END) AS with_client
            FROM {fqtn}
            """
            df = safe_read_sql(q, engine)
            r = df.iloc[0]
            distinct_clients = int(r['distinct_clients'])
            missing_client = int(r['missing_client'])
            with_client = int(r['with_client'])
            client_coverage = round(with_client / total_rows * 100, 2) if total_rows > 0 else 0

            client_recurrence_records.append({'feature': 'client_key_coverage', 'value': client_coverage, 'unit': '%', 'note': f'{with_client:,} of {total_rows:,}'})
            client_recurrence_records.append({'feature': 'distinct_clients', 'value': distinct_clients, 'unit': '', 'note': ''})
            print(f"  distinct_clients={distinct_clients:,} | with_client={with_client:,} ({client_coverage}%) | missing={missing_client:,}")

            # Claims per client distribution
            q2 = f"""
            SELECT
                claims_per_client,
                COUNT(*) AS client_count
            FROM (
                SELECT "{client_col}", COUNT(*) AS claims_per_client
                FROM {fqtn}
                WHERE "{client_col}" IS NOT NULL AND "{client_col}" <> 0
                GROUP BY "{client_col}"
            ) sub
            GROUP BY claims_per_client
            ORDER BY claims_per_client
            """
            dist_df = safe_read_sql(q2, engine)
            clients_with_multi = int(dist_df[dist_df['claims_per_client'] > 1]['client_count'].sum()) if not dist_df.empty else 0
            pct_multi = round(clients_with_multi / distinct_clients * 100, 2) if distinct_clients > 0 else 0
            client_recurrence_records.append({'feature': 'clients_with_more_than_1_claim', 'value': clients_with_multi, 'unit': '', 'note': f'{pct_multi}% of clients'})
            print(f"  clients_with_>1_claim={clients_with_multi:,} ({pct_multi}%)")

            if client_coverage >= 90:
                CLIENT_RECURRENCE_STATUS = STATUS_READY
            elif client_coverage >= 60:
                CLIENT_RECURRENCE_STATUS = STATUS_PARTIAL
            else:
                CLIENT_RECURRENCE_STATUS = STATUS_NOT_READY

            client_recurrence_records.append({'feature': 'client_recurrence_status', 'value': CLIENT_RECURRENCE_STATUS, 'unit': '', 'note': f'Coverage {client_coverage}%'})
            print(f"  → Client recurrence status: {CLIENT_RECURRENCE_STATUS}")

        except Exception as e:
            CLIENT_RECURRENCE_STATUS = STATUS_TO_AUDIT
            client_recurrence_records.append({'feature': 'client_recurrence_audit', 'value': 'ERROR', 'unit': '', 'note': str(e)})
            print(f"  [ERROR] {e}")

    client_rec_df = pd.DataFrame(client_recurrence_records)
    out_path = OUTPUT_DIR / 'claim_scoring_client_recurrence_readiness.csv'
    client_rec_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"\n[EXPORT] Client recurrence readiness saved: {out_path}")
    display(client_rec_df)

print(f"\nClient recurrence status: {CLIENT_RECURRENCE_STATUS}")

[INFO] Client recurrence readiness — client_col='client_sk', claim_date_col='date_survenance_sk'
  distinct_clients=114,188 | with_client=381,893 (100.0%) | missing=0
  clients_with_>1_claim=82,371 (72.14%)
  → Client recurrence status: READY

[EXPORT] Client recurrence readiness saved: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\data_readiness\claim_scoring_client_recurrence_readiness.csv


,feature,value,unit,note
0,client_key_coverage,100.0000,%,"381,893 of 381,893"
1,distinct_clients,114188,,
2,clients_with_more_than_1_claim,82371,,72.14% of clients
3,client_recurrence_status,READY,,Coverage 100.0%



Client recurrence status: READY


---
## Section 11 — Préparation de la récurrence véhicule

In [10]:
# =============================================================================
# SECTION 11 — VEHICLE RECURRENCE READINESS
# =============================================================================

vehicle_recurrence_records = []
VEHICLE_RECURRENCE_STATUS = STATUS_TO_AUDIT

if not AUDIT_STATE['db_connected'] or not AUDIT_STATE['best_claim_table']:
    print("[SKIP] No claim table. Skipping vehicle recurrence readiness.")
else:
    schema = AUDIT_STATE['best_claim_schema']
    table = AUDIT_STATE['best_claim_table']
    fqtn = f'"{schema}"."{table}"'
    vehicle_col = AUDIT_STATE.get('vehicle_key_col')
    claim_date_col = AUDIT_STATE.get('claim_date_col')
    total_rows = AUDIT_STATE.get('total_claim_rows', 0)

    print(f"[INFO] Vehicle recurrence readiness — vehicle_col='{vehicle_col}', claim_date_col='{claim_date_col}'")

    if not vehicle_col:
        VEHICLE_RECURRENCE_STATUS = STATUS_NOT_AVAILABLE
        vehicle_recurrence_records.append({'feature': 'vehicle_recurrence', 'status': STATUS_NOT_AVAILABLE, 'note': 'No vehicle key column found'})
        print("  [WARN] Vehicle key column not found → NOT_AVAILABLE")
    else:
        try:
            q = f"""
            SELECT
                COUNT(DISTINCT NULLIF("{vehicle_col}", 0)) AS distinct_vehicles,
                COUNT(*) AS total_claims,
                SUM(CASE WHEN "{vehicle_col}" IS NULL OR "{vehicle_col}" = 0 THEN 1 ELSE 0 END) AS missing_vehicle,
                SUM(CASE WHEN "{vehicle_col}" IS NOT NULL AND "{vehicle_col}" <> 0 THEN 1 ELSE 0 END) AS with_vehicle
            FROM {fqtn}
            """
            df = safe_read_sql(q, engine)
            r = df.iloc[0]
            distinct_vehicles = int(r['distinct_vehicles'])
            missing_vehicle = int(r['missing_vehicle'])
            with_vehicle = int(r['with_vehicle'])
            vehicle_coverage = round(with_vehicle / total_rows * 100, 2) if total_rows > 0 else 0

            vehicle_recurrence_records.append({'feature': 'vehicle_key_coverage', 'value': vehicle_coverage, 'unit': '%', 'note': f'{with_vehicle:,} of {total_rows:,}'})
            vehicle_recurrence_records.append({'feature': 'distinct_vehicles', 'value': distinct_vehicles, 'unit': '', 'note': ''})
            print(f"  distinct_vehicles={distinct_vehicles:,} | with_vehicle={with_vehicle:,} ({vehicle_coverage}%) | missing={missing_vehicle:,}")

            q2 = f"""
            SELECT
                claims_per_vehicle,
                COUNT(*) AS vehicle_count
            FROM (
                SELECT "{vehicle_col}", COUNT(*) AS claims_per_vehicle
                FROM {fqtn}
                WHERE "{vehicle_col}" IS NOT NULL AND "{vehicle_col}" <> 0
                GROUP BY "{vehicle_col}"
            ) sub
            GROUP BY claims_per_vehicle
            ORDER BY claims_per_vehicle
            """
            dist_df = safe_read_sql(q2, engine)
            vehicles_with_multi = int(dist_df[dist_df['claims_per_vehicle'] > 1]['vehicle_count'].sum()) if not dist_df.empty else 0
            pct_multi = round(vehicles_with_multi / distinct_vehicles * 100, 2) if distinct_vehicles > 0 else 0
            vehicle_recurrence_records.append({'feature': 'vehicles_with_more_than_1_claim', 'value': vehicles_with_multi, 'unit': '', 'note': f'{pct_multi}% of vehicles'})
            print(f"  vehicles_with_>1_claim={vehicles_with_multi:,} ({pct_multi}%)")

            if vehicle_coverage >= 90:
                VEHICLE_RECURRENCE_STATUS = STATUS_READY
            elif vehicle_coverage >= 60:
                VEHICLE_RECURRENCE_STATUS = STATUS_PARTIAL
            else:
                VEHICLE_RECURRENCE_STATUS = STATUS_NOT_READY

            vehicle_recurrence_records.append({'feature': 'vehicle_recurrence_status', 'value': VEHICLE_RECURRENCE_STATUS, 'unit': '', 'note': f'Coverage {vehicle_coverage}%'})
            print(f"  → Vehicle recurrence status: {VEHICLE_RECURRENCE_STATUS}")

        except Exception as e:
            VEHICLE_RECURRENCE_STATUS = STATUS_TO_AUDIT
            vehicle_recurrence_records.append({'feature': 'vehicle_recurrence_audit', 'value': 'ERROR', 'unit': '', 'note': str(e)})
            print(f"  [ERROR] {e}")

    veh_rec_df = pd.DataFrame(vehicle_recurrence_records)
    out_path = OUTPUT_DIR / 'claim_scoring_vehicle_recurrence_readiness.csv'
    veh_rec_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"\n[EXPORT] Vehicle recurrence readiness saved: {out_path}")
    display(veh_rec_df)

print(f"\nVehicle recurrence status: {VEHICLE_RECURRENCE_STATUS}")

[INFO] Vehicle recurrence readiness — vehicle_col='vehicule_sk', claim_date_col='date_survenance_sk'
  distinct_vehicles=232 | with_vehicle=381,893 (100.0%) | missing=0
  vehicles_with_>1_claim=192 (82.76%)
  → Vehicle recurrence status: READY

[EXPORT] Vehicle recurrence readiness saved: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\data_readiness\claim_scoring_vehicle_recurrence_readiness.csv


,feature,value,unit,note
0,vehicle_key_coverage,100.0000,%,"381,893 of 381,893"
1,distinct_vehicles,232,,
2,vehicles_with_more_than_1_claim,192,,82.76% of vehicles
3,vehicle_recurrence_status,READY,,Coverage 100.0%



Vehicle recurrence status: READY


---
## Section 12 — Préparation tiers / conducteur

> **Note métier :** Si les données tiers ou conducteur sont incomplètes, leur absence affecte le **niveau de confiance** et non le score d'attention. Cette famille est classée **P3** dans la feuille de route.

In [11]:
# =============================================================================
# SECTION 12 — THIRD-PARTY / DRIVER READINESS
# Search for candidate third-party and driver tables/columns.
# =============================================================================

tp_driver_records = []
TP_DRIVER_STATUS = STATUS_TO_AUDIT

if not AUDIT_STATE['db_connected'] or table_inventory_df.empty:
    print("[SKIP] No database connection or table inventory. Skipping third-party/driver readiness.")
else:
    TP_KEYWORDS = ['tiers', 'third', 'adversaire', 'advers', 'conducteur', 'driver', 'chauff']
    TP_COL_KEYWORDS = ['tiers', 'numt', 'third', 'adversaire', 'conducteur', 'driver', 'id_tiers', 'tiers_sk']

    tp_mask = table_inventory_df['table_name'].str.lower().apply(
        lambda n: any(kw in n for kw in TP_KEYWORDS)
    )
    tp_tables = table_inventory_df[tp_mask].copy()

    print(f"[INFO] Found {len(tp_tables)} candidate third-party/driver table(s):")
    if not tp_tables.empty:
        print(tp_tables[['table_schema', 'table_name', 'row_count_estimate']].to_string(index=False))

    for _, row in tp_tables.iterrows():
        schema = row['table_schema']
        table = row['table_name']
        try:
            col_df = safe_read_sql(
                f"SELECT column_name FROM information_schema.columns WHERE table_schema='{schema}' AND table_name='{table}'",
                engine
            )
            all_cols = col_df['column_name'].str.lower().tolist()
            detected_tp_cols = [c for c in all_cols if any(kw in c for kw in TP_COL_KEYWORDS)]

            # Check link to claim table
            claim_id_col = AUDIT_STATE.get('claim_id_col')
            link_col_found = None
            if claim_id_col:
                claim_kw = ['sinistre', 'claim', 'snt', 'numsnt', 'numsini']
                link_cols = [c for c in all_cols if any(kw in c for kw in claim_kw)]
                link_col_found = link_cols[0] if link_cols else None

            tp_driver_records.append({
                'table_schema': schema,
                'table_name': table,
                'row_count': row.get('row_count_estimate', -1),
                'detected_tp_columns': ', '.join(detected_tp_cols) or 'NONE',
                'link_to_claim_col': link_col_found or 'NOT_FOUND',
                'readiness_note': 'Link possible' if link_col_found else 'No claim link found',
            })
        except Exception as e:
            tp_driver_records.append({'table_schema': schema, 'table_name': table, 'row_count': -1,
                                      'detected_tp_columns': 'ERROR', 'link_to_claim_col': 'ERROR', 'readiness_note': str(e)})

    # Also check for TP columns within the main claim table
    if AUDIT_STATE.get('best_claim_table'):
        schema = AUDIT_STATE['best_claim_schema']
        table = AUDIT_STATE['best_claim_table']
        try:
            col_df = safe_read_sql(
                f"SELECT column_name FROM information_schema.columns WHERE table_schema='{schema}' AND table_name='{table}'",
                engine
            )
            all_cols = col_df['column_name'].str.lower().tolist()
            inline_tp = [c for c in all_cols if any(kw in c for kw in TP_COL_KEYWORDS)]
            if inline_tp:
                tp_driver_records.append({
                    'table_schema': schema,
                    'table_name': f'{table} (CLAIM TABLE)',
                    'row_count': AUDIT_STATE.get('total_claim_rows', -1),
                    'detected_tp_columns': ', '.join(inline_tp),
                    'link_to_claim_col': 'INLINE',
                    'readiness_note': 'Third-party columns found inline in claim table',
                })
                print(f"  [INFO] Inline TP columns in claim table: {inline_tp}")
        except:
            pass

    if not tp_driver_records:
        tp_driver_records.append({'table_schema': 'N/A', 'table_name': 'N/A', 'row_count': 0,
                                   'detected_tp_columns': 'NONE', 'link_to_claim_col': 'NOT_FOUND',
                                   'readiness_note': 'No third-party or driver table/column detected'})
        TP_DRIVER_STATUS = STATUS_NOT_AVAILABLE
    else:
        TP_DRIVER_STATUS = STATUS_PARTIAL  # Default: partial unless confirmed linkable

    tp_df = pd.DataFrame(tp_driver_records)
    out_path = OUTPUT_DIR / 'claim_scoring_third_party_driver_readiness.csv'
    tp_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"\n[EXPORT] Third-party/driver readiness saved: {out_path}")
    display(tp_df)

print(f"\nThird-party/driver status: {TP_DRIVER_STATUS} (expected P3 — not required for V1)")

[INFO] Found 2 candidate third-party/driver table(s):
table_schema     table_name  row_count_estimate
         dwh dim_conducteur              161523
         dwh      dim_tiers              167175
  [INFO] Inline TP columns in claim table: ['conducteur_sk', 'tiers_sk']

[EXPORT] Third-party/driver readiness saved: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\data_readiness\claim_scoring_third_party_driver_readiness.csv


,table_schema,table_name,row_count,detected_tp_columns,link_to_claim_col,readiness_note
0,dwh,dim_conducteur,161523,"conducteur_sk, nom_conducteur, date_naissance_...",NOT_FOUND,No claim link found
1,dwh,dim_tiers,167175,"tiers_sk, nom_tiers, immatriculation_vehicule_...",numero_sinistre_tiers,Link possible
2,dwh,fact_sinistre (CLAIM TABLE),381893,"conducteur_sk, tiers_sk",INLINE,Third-party columns found inline in claim table



Third-party/driver status: PARTIAL (expected P3 — not required for V1)


---
## Section 13 — Préparation de la chronologie

> **Note métier :** Les features de chronologie nécessitent la disponibilité des tables contrat et garantie avec leurs dates. Un délai court entre début de contrat et sinistre constitue un **contexte à analyser**, pas une conclusion automatique.

In [12]:
# =============================================================================
# SECTION 13 — CHRONOLOGY READINESS
# =============================================================================

chronology_records = []
CHRONOLOGY_STATUS = STATUS_TO_AUDIT

if not AUDIT_STATE['db_connected'] or table_inventory_df.empty:
    print("[SKIP] No database connection or table inventory. Skipping chronology readiness.")
else:
    CONTRACT_KEYWORDS = ['contrat', 'contract', 'production', 'police', 'avenant']
    CONTRACT_DATE_COLS = ['date_effet', 'date_debut', 'date_fin', 'date_modification', 'date_avenant',
                          'date_echeance', 'dteffet', 'date_souscription']
    GUARANTEE_KEYWORDS = ['garantie', 'guarantee', 'grnt', 'couverture']

    # Search contract tables
    contract_mask = table_inventory_df['table_name'].str.lower().apply(
        lambda n: any(kw in n for kw in CONTRACT_KEYWORDS)
    )
    contract_tables = table_inventory_df[contract_mask].copy()
    print(f"[INFO] Found {len(contract_tables)} candidate contract table(s):")
    if not contract_tables.empty:
        print(contract_tables[['table_schema', 'table_name', 'row_count_estimate']].to_string(index=False))

    # Evaluate each chronology feature
    CHRONOLOGY_FEATURES = [
        ('days_contract_start_to_claim', 'Requires: claim_date + contract start date (joinable)'),
        ('days_claim_to_declaration', 'Requires: claim_date + declaration_date (in claim table)'),
        ('recent_contract_change_flag', 'Requires: contract modification history table'),
        ('recent_guarantee_change_flag', 'Requires: guarantee modification history table'),
        ('claim_before_contract_start_flag', 'Requires: claim_date + contract start date (joinable)'),
        ('claim_after_recent_update_flag', 'Derived from recent_contract_change_flag + recent_guarantee_change_flag'),
    ]

    # days_claim_to_declaration — check in claim table
    has_claim_date = bool(AUDIT_STATE.get('claim_date_col'))
    has_decl_date = bool(AUDIT_STATE.get('declaration_date_col'))

    # Contract tables column scan
    contract_date_found = False
    contract_link_col = None
    for _, row in contract_tables.iterrows():
        s, t = row['table_schema'], row['table_name']
        try:
            col_df = safe_read_sql(
                f"SELECT column_name FROM information_schema.columns WHERE table_schema='{s}' AND table_name='{t}'",
                engine
            )
            cols = col_df['column_name'].str.lower().tolist()
            found_dates = [c for c in cols if any(kw in c for kw in CONTRACT_DATE_COLS)]
            if found_dates:
                contract_date_found = True
                # Check if joinable to claim table
                contract_claim_links = ['numcnt', 'contract_sk', 'id_contrat', 'contrat_sk', 'numsnt', 'sinistre_sk']
                link_cols = [c for c in cols if any(kw in c for kw in contract_claim_links)]
                contract_link_col = link_cols[0] if link_cols else None
                chronology_records.append({
                    'source': f'{s}.{t}',
                    'date_columns_found': ', '.join(found_dates),
                    'link_col': contract_link_col or 'NOT_FOUND',
                    'note': 'Contract date source candidate'
                })
        except Exception as e:
            chronology_records.append({'source': f'{s}.{t}', 'date_columns_found': 'ERROR',
                                        'link_col': 'ERROR', 'note': str(e)})

    # Feature-level readiness
    feature_records = []
    for feat, requires in CHRONOLOGY_FEATURES:
        if feat == 'days_claim_to_declaration':
            status = STATUS_READY if (has_claim_date and has_decl_date) else (STATUS_PARTIAL if has_claim_date else STATUS_NOT_READY)
            note = 'claim_date and declaration_date both found' if (has_claim_date and has_decl_date) else 'Missing date columns'
        elif feat in ('days_contract_start_to_claim', 'claim_before_contract_start_flag'):
            status = STATUS_PARTIAL if (has_claim_date and contract_date_found) else STATUS_NOT_READY
            note = 'Contract date source found, join required' if contract_date_found else 'No contract start date found'
        elif feat in ('recent_contract_change_flag', 'recent_guarantee_change_flag', 'claim_after_recent_update_flag'):
            status = STATUS_TO_AUDIT
            note = 'Requires modification history — availability to confirm'
        else:
            status = STATUS_TO_AUDIT
            note = 'Requires further audit'

        feature_records.append({'feature': feat, 'requires': requires, 'status': status, 'note': note})

    # Overall chronology status
    statuses = [r['status'] for r in feature_records]
    if statuses.count(STATUS_READY) >= 2:
        CHRONOLOGY_STATUS = STATUS_PARTIAL  # Some features ready
    elif all(s in (STATUS_NOT_READY, STATUS_NOT_AVAILABLE) for s in statuses):
        CHRONOLOGY_STATUS = STATUS_NOT_READY
    else:
        CHRONOLOGY_STATUS = STATUS_PARTIAL

    all_records = chronology_records + feature_records
    chrono_df = pd.DataFrame(feature_records)
    out_path = OUTPUT_DIR / 'claim_scoring_chronology_readiness.csv'
    chrono_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"\n[EXPORT] Chronology readiness saved: {out_path}")
    display(chrono_df)

print(f"\nChronology status: {CHRONOLOGY_STATUS}")

[INFO] Found 3 candidate contract table(s):
table_schema     table_name  row_count_estimate
         dwh    dim_contrat              447585
         dwh   fact_contrat              585514
     staging stg_production                   0

[EXPORT] Chronology readiness saved: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\data_readiness\claim_scoring_chronology_readiness.csv


,feature,requires,status,note
0,days_contract_start_to_claim,Requires: claim_date + contract start date (jo...,PARTIAL,"Contract date source found, join required"
1,days_claim_to_declaration,Requires: claim_date + declaration_date (in cl...,READY,claim_date and declaration_date both found
2,recent_contract_change_flag,Requires: contract modification history table,TO_AUDIT,Requires modification history — availability t...
3,recent_guarantee_change_flag,Requires: guarantee modification history table,TO_AUDIT,Requires modification history — availability t...
4,claim_before_contract_start_flag,Requires: claim_date + contract start date (jo...,PARTIAL,"Contract date source found, join required"
5,claim_after_recent_update_flag,Derived from recent_contract_change_flag + rec...,TO_AUDIT,Requires modification history — availability t...



Chronology status: PARTIAL


---
## Section 14 — Préparation des données géographiques (GEO)

> **Règle critique :** Une géographie manquante ou UNKNOWN est un **problème de qualité des données**, pas un signal de suspicion. Les features GEO sont classées **P2** — à activer uniquement après stabilisation de l'ETL GEO.

In [13]:
# =============================================================================
# SECTION 14 — GEO READINESS
# =============================================================================

geo_records = []
GEO_STATUS = STATUS_TO_AUDIT

if not AUDIT_STATE['db_connected'] or table_inventory_df.empty:
    print("[SKIP] Skipping GEO readiness.")
else:
    GEO_TABLE_KEYWORDS = ['geo', 'region', 'gouvern', 'deleg', 'localit', 'adresse', 'agence']
    GEO_COL_KEYWORDS = ['geo_sk', 'region', 'gouvernorat', 'delegation', 'localite', 'lieu', 'adresse', 'wilaya', 'code_postal']
    UNKNOWN_LABELS = ['UNKNOWN', 'INCONNU', 'N/A', '', 'NON DEFINI', '?']

    geo_mask = table_inventory_df['table_name'].str.lower().apply(
        lambda n: any(kw in n for kw in GEO_TABLE_KEYWORDS)
    )
    geo_tables = table_inventory_df[geo_mask].copy()
    print(f"[INFO] Found {len(geo_tables)} candidate GEO/agency table(s):")
    if not geo_tables.empty:
        print(geo_tables[['table_schema', 'table_name', 'row_count_estimate']].to_string(index=False))

    # Check claim_geo_sk in claim table
    claim_geo = AUDIT_STATE.get('geo_key_col')
    if claim_geo and AUDIT_STATE.get('best_claim_table'):
        schema = AUDIT_STATE['best_claim_schema']
        table = AUDIT_STATE['best_claim_table']
        fqtn = f'"{schema}"."{table}"'
        total_rows = AUDIT_STATE.get('total_claim_rows', 0)
        try:
            q = f"""
            SELECT
                COUNT("{claim_geo}") AS geo_non_null,
                COUNT(*) - COUNT("{claim_geo}") AS geo_null,
                COUNT(CASE WHEN CAST("{claim_geo}" AS TEXT) IN ('UNKNOWN','INCONNU','','N/A') THEN 1 END) AS geo_unknown
            FROM {fqtn}
            """
            df = safe_read_sql(q, engine)
            r = df.iloc[0]
            geo_non_null = int(r['geo_non_null'])
            geo_null = int(r['geo_null'])
            geo_unknown = int(r['geo_unknown'])
            geo_coverage = round(geo_non_null / total_rows * 100, 2) if total_rows > 0 else 0

            geo_records.append({'check': 'claim_geo_col', 'value': claim_geo, 'note': ''})
            geo_records.append({'check': 'claim_geo_non_null', 'value': geo_non_null, 'note': f'{geo_coverage}% coverage'})
            geo_records.append({'check': 'claim_geo_null', 'value': geo_null, 'note': ''})
            geo_records.append({'check': 'claim_geo_unknown_labels', 'value': geo_unknown, 'note': 'Data quality limitation — not suspicion'})
            print(f"  claim_geo='{claim_geo}': non_null={geo_non_null:,} ({geo_coverage}%) | null={geo_null:,} | UNKNOWN_labels={geo_unknown:,}")

            if geo_coverage >= 80 and geo_unknown < total_rows * 0.2:
                GEO_STATUS = STATUS_PARTIAL  # PARTIAL until ETL GEO explicitly confirmed stable
            elif geo_coverage >= 50:
                GEO_STATUS = STATUS_PARTIAL
            else:
                GEO_STATUS = STATUS_NOT_READY
        except Exception as e:
            geo_records.append({'check': 'claim_geo_audit', 'value': 'ERROR', 'note': str(e)})
            print(f"  [ERROR] GEO audit: {e}")
    else:
        geo_records.append({'check': 'claim_geo_col', 'value': 'NOT_AVAILABLE', 'note': 'No GEO column found in claim table'})
        GEO_STATUS = STATUS_NOT_READY

    # Scan GEO tables for UNKNOWN counts
    for _, row in geo_tables.iterrows():
        s, t = row['table_schema'], row['table_name']
        if 'geo' in t.lower() or 'region' in t.lower():
            try:
                col_df = safe_read_sql(
                    f"SELECT column_name FROM information_schema.columns WHERE table_schema='{s}' AND table_name='{t}'",
                    engine
                )
                geo_cols = [c for c in col_df['column_name'].str.lower().tolist() if any(kw in c for kw in GEO_COL_KEYWORDS)]
                geo_records.append({'check': f'geo_table_{s}.{t}', 'value': ', '.join(geo_cols) or 'NONE', 'note': 'GEO table candidate'})
            except:
                pass

    geo_df = pd.DataFrame(geo_records)
    out_path = OUTPUT_DIR / 'claim_scoring_geo_readiness.csv'
    geo_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"\n[EXPORT] GEO readiness saved: {out_path}")
    display(geo_df)

print(f"\nGEO status: {GEO_STATUS} (P2 — activate after ETL GEO stabilization)")

[INFO] Found 1 candidate GEO/agency table(s):
table_schema table_name  row_count_estimate
         dwh    dim_geo                1856

[EXPORT] GEO readiness saved: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\data_readiness\claim_scoring_geo_readiness.csv


,check,value,note
0,claim_geo_col,NOT_AVAILABLE,No GEO column found in claim table
1,geo_table_dwh.dim_geo,"geo_sk, region, gouvernorat, localite, adresse...",GEO table candidate



GEO status: NOT_READY (P2 — activate after ETL GEO stabilization)


---
## Section 15 — Préparation de la liaison VHS

> **Positionnement VHS :** Le VHS est un signal technique contextuel, limité à 5 points sur 100 dans la V1. Son absence ne bloque pas le calcul du score. Cette famille est classée **P2**.

In [14]:
# =============================================================================
# SECTION 15 — VHS LINKAGE READINESS
# Uses the known VHS mart tables from the finalized VHS module.
# =============================================================================

vhs_records = []
VHS_STATUS = STATUS_TO_AUDIT

if not AUDIT_STATE['db_connected'] or table_inventory_df.empty:
    print("[SKIP] Skipping VHS linkage readiness.")
else:
    VHS_CANDIDATE_TABLES = [
        ('mart', 'fact_vhs_score'),
        ('mart', 'fact_vhs_scores'),
        ('mart', 'fact_vhs_penalty_detail'),
        ('mart', 'vhs_scores'),
    ]

    # Also search table_inventory
    vhs_mask = table_inventory_df['table_name'].str.lower().apply(
        lambda n: 'vhs' in n or 'staffim' in n or 'inspection' in n
    )
    vhs_from_inventory = table_inventory_df[vhs_mask][['table_schema', 'table_name']].values.tolist()
    VHS_CANDIDATE_TABLES = list(set(VHS_CANDIDATE_TABLES + [(s, t) for s, t in vhs_from_inventory]))

    VHS_KEY_COLS = ['vehicle_sk', 'vehicule_sk', 'immatriculation', 'matricule']
    VHS_DATE_COLS = ['inspection_date', 'date_inspection', 'date_visite', 'date_controle']
    VHS_SCORE_COLS = ['vhs_score', 'score', 'total_score', 'score_total']
    VHS_GRADE_COLS = ['grade', 'vhs_grade', 'attention_level', 'vhs_attention_level', 'niveau']

    found_vhs_table = None
    found_vhs_schema = None
    found_vhs_vehicle_col = None
    found_vhs_date_col = None

    for schema, table in VHS_CANDIDATE_TABLES:
        try:
            col_df = safe_read_sql(
                f"SELECT column_name FROM information_schema.columns WHERE table_schema='{schema}' AND table_name='{table}'",
                engine
            )
            if col_df.empty:
                continue
            cols = col_df['column_name'].str.lower().tolist()
            veh_cols = [c for c in cols if any(kw in c for kw in VHS_KEY_COLS)]
            date_cols = [c for c in cols if any(kw in c for kw in VHS_DATE_COLS)]
            score_cols = [c for c in cols if any(kw in c for kw in VHS_SCORE_COLS)]
            grade_cols = [c for c in cols if any(kw in c for kw in VHS_GRADE_COLS)]

            row_q = f'SELECT COUNT(*) AS cnt FROM "{schema}"."{table}"'
            row_cnt = int(safe_read_sql(row_q, engine)['cnt'].iloc[0])

            vhs_records.append({
                'vhs_table': f'{schema}.{table}',
                'row_count': row_cnt,
                'vehicle_key_col': ', '.join(veh_cols) or 'NONE',
                'date_col': ', '.join(date_cols) or 'NONE',
                'score_col': ', '.join(score_cols) or 'NONE',
                'grade_col': ', '.join(grade_cols) or 'NONE',
                'note': 'VHS table found',
            })
            if not found_vhs_table and veh_cols:
                found_vhs_table = table
                found_vhs_schema = schema
                found_vhs_vehicle_col = veh_cols[0]
                found_vhs_date_col = date_cols[0] if date_cols else None

            print(f"  [VHS] {schema}.{table}: {row_cnt:,} rows | veh_cols={veh_cols} | date_cols={date_cols} | score_cols={score_cols}")
        except Exception as e:
            # Table does not exist or error — skip silently
            pass

    # Estimate VHS linkage rate
    if found_vhs_table and AUDIT_STATE.get('best_claim_table') and found_vhs_vehicle_col:
        claim_schema = AUDIT_STATE['best_claim_schema']
        claim_table = AUDIT_STATE['best_claim_table']
        vehicle_col = AUDIT_STATE.get('vehicle_key_col')
        total_rows = AUDIT_STATE.get('total_claim_rows', 0)

        if vehicle_col:
            try:
                linkage_q = f"""
                SELECT COUNT(DISTINCT c."{vehicle_col}") AS claims_with_vhs
                FROM "{claim_schema}"."{claim_table}" c
                INNER JOIN "{found_vhs_schema}"."{found_vhs_table}" v
                    ON c."{vehicle_col}" = v."{found_vhs_vehicle_col}"
                WHERE c."{vehicle_col}" IS NOT NULL
                """
                ldf = safe_read_sql(linkage_q, engine)
                claims_with_vhs = int(ldf['claims_with_vhs'].iloc[0])
                linkage_rate = round(claims_with_vhs / total_rows * 100, 2) if total_rows > 0 else 0
                vhs_records.append({
                    'vhs_table': 'LINKAGE_RATE',
                    'row_count': claims_with_vhs,
                    'vehicle_key_col': vehicle_col,
                    'date_col': '',
                    'score_col': '',
                    'grade_col': '',
                    'note': f'VHS linkage rate: {linkage_rate}%',
                })
                print(f"  VHS linkage rate: {linkage_rate}%")

                if linkage_rate >= 50:
                    VHS_STATUS = STATUS_PARTIAL
                elif linkage_rate >= 20:
                    VHS_STATUS = STATUS_PARTIAL
                else:
                    VHS_STATUS = STATUS_NOT_READY
            except Exception as e:
                vhs_records.append({'vhs_table': 'LINKAGE_RATE', 'row_count': 0, 'vehicle_key_col': '',
                                    'date_col': '', 'score_col': '', 'grade_col': '', 'note': f'ERROR: {e}'})
                print(f"  [ERROR] VHS linkage: {e}")
    elif found_vhs_table:
        VHS_STATUS = STATUS_PARTIAL
    else:
        vhs_records.append({'vhs_table': 'NOT_FOUND', 'row_count': 0, 'vehicle_key_col': 'N/A',
                             'date_col': 'N/A', 'score_col': 'N/A', 'grade_col': 'N/A',
                             'note': 'No VHS table found — VHS module may need deployment or different schema'})
        VHS_STATUS = STATUS_NOT_AVAILABLE

    vhs_df = pd.DataFrame(vhs_records)
    out_path = OUTPUT_DIR / 'claim_scoring_vhs_linkage_readiness.csv'
    vhs_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"\n[EXPORT] VHS linkage readiness saved: {out_path}")
    display(vhs_df)

print(f"\nVHS status: {VHS_STATUS} (P2 — absence does not block V1 scoring)")

  [VHS] mart.fact_vhs_penalty_detail: 33,916 rows | veh_cols=['vehicule_sk', 'immatriculation_norm'] | date_cols=['date_inspection_sk'] | score_cols=[]
  [VHS] dwh.fact_inspection_vehicule: 286 rows | veh_cols=['fact_inspection_vehicule_sk', 'immatriculation_norm', 'vehicule_sk'] | date_cols=['date_inspection_sk'] | score_cols=[]
  [VHS] dwh.fact_inspection_checkpoint: 12,298 rows | veh_cols=['vehicule_sk', 'immatriculation_norm'] | date_cols=['date_inspection_sk'] | score_cols=[]
  [VHS] staging.stg_inspection: 286 rows | veh_cols=['immatriculation'] | date_cols=['date_inspection'] | score_cols=['score_etat_vehicule']
  [VHS] mart.fact_vhs_score: 2,002 rows | veh_cols=['vehicule_sk', 'immatriculation_norm'] | date_cols=['date_inspection_sk'] | score_cols=['vhs_score_sk', 'vhs_raw_score', 'vhs_final_score', 'safety_score', 'functional_score', 'cosmetic_score', 'nb_checkpoints_scored']
  VHS linkage rate: 0.06%

[EXPORT] VHS linkage readiness saved: C:\Users\wiem\Downloads\Projet PFE\IR

,vhs_table,row_count,vehicle_key_col,date_col,score_col,grade_col,note
0,mart.fact_vhs_penalty_detail,33916,"vehicule_sk, immatriculation_norm",date_inspection_sk,NONE,NONE,VHS table found
1,dwh.fact_inspection_vehicule,286,"fact_inspection_vehicule_sk, immatriculation_n...",date_inspection_sk,NONE,NONE,VHS table found
2,dwh.fact_inspection_checkpoint,12298,"vehicule_sk, immatriculation_norm",date_inspection_sk,NONE,NONE,VHS table found
3,staging.stg_inspection,286,immatriculation,date_inspection,score_etat_vehicule,"niveau_etat_vehicule, sous_le_capot_controle_d...",VHS table found
4,mart.fact_vhs_score,2002,"vehicule_sk, immatriculation_norm",date_inspection_sk,"vhs_score_sk, vhs_raw_score, vhs_final_score, ...",safety_grade,VHS table found
5,LINKAGE_RATE,232,vehicule_sk,,,,VHS linkage rate: 0.06%



VHS status: NOT_READY (P2 — absence does not block V1 scoring)


---
## Section 16 — Préparation de la qualité des données et confiance

> **Principe fondamental :** Un problème de qualité des données est une **limitation d'analyse**, pas un signal de suspicion. Les features de qualité affectent uniquement le niveau de confiance.

In [15]:
# =============================================================================
# SECTION 16 — DATA QUALITY AND CONFIDENCE READINESS
# Exploratory computation of confidence-related metrics.
# =============================================================================

confidence_records = []
CONFIDENCE_STATUS = STATUS_TO_AUDIT

if not AUDIT_STATE['db_connected'] or not AUDIT_STATE['best_claim_table']:
    print("[SKIP] Skipping confidence readiness.")
else:
    schema = AUDIT_STATE['best_claim_schema']
    table = AUDIT_STATE['best_claim_table']
    fqtn = f'"{schema}"."{table}"'
    total_rows = AUDIT_STATE.get('total_claim_rows', 0)

    client_col = AUDIT_STATE.get('client_key_col')
    vehicle_col = AUDIT_STATE.get('vehicle_key_col')
    contract_col = AUDIT_STATE.get('contract_key_col')
    agency_col = AUDIT_STATE.get('agency_key_col')
    geo_col = AUDIT_STATE.get('geo_key_col')
    claim_date_col = AUDIT_STATE.get('claim_date_col')
    amount_col = AUDIT_STATE.get('amount_col')

    print(f"[INFO] Data quality / confidence readiness on: {fqtn}")

    # Build dynamic SELECT with only available columns
    select_parts = []
    def missing_key_expr(col):
        return f'SUM(CASE WHEN "{col}" IS NULL OR "{col}" = 0 THEN 1 ELSE 0 END)'

    if client_col:   select_parts.append(f'{missing_key_expr(client_col)} AS missing_client_count')
    if vehicle_col:  select_parts.append(f'{missing_key_expr(vehicle_col)} AS missing_vehicle_count')
    if contract_col: select_parts.append(f'{missing_key_expr(contract_col)} AS missing_contract_count')
    if agency_col:   select_parts.append(f'{missing_key_expr(agency_col)} AS missing_agency_count')
    if geo_col:      select_parts.append(f'{missing_key_expr(geo_col)} AS missing_geo_count')
    if claim_date_col:
        select_parts.append(f'SUM(CASE WHEN "{claim_date_col}" > 0 AND "{claim_date_col}" < 20190101 THEN 1 ELSE 0 END) AS migration_2019_count')

    if not select_parts:
        confidence_records.append({'metric': 'data_quality_audit', 'value': 'NO_COLUMNS', 'note': 'No key columns found for quality check'})
    else:
        try:
            q = f"SELECT {', '.join(select_parts)} FROM {fqtn}"
            df = safe_read_sql(q, engine)
            r = df.iloc[0]

            null_flags = []
            for col_label, col_name in [
                ('missing_client_count', 'missing_client'),
                ('missing_vehicle_count', 'missing_vehicle'),
                ('missing_contract_count', 'missing_contract'),
                ('missing_agency_count', 'missing_agency'),
                ('missing_geo_count', 'missing_geo'),
            ]:
                if col_label in r.index:
                    val = int(r[col_label])
                    pct = round(val / total_rows * 100, 2) if total_rows > 0 else 0
                    confidence_records.append({'metric': f'{col_name}_flag_count', 'value': val, 'note': f'{pct}% of claims'})
                    null_flags.append(pct)
                    print(f"  {col_name}: {val:,} ({pct}%)")

            if 'migration_2019_count' in r.index:
                m2019 = int(r['migration_2019_count'])
                pct_m = round(m2019 / total_rows * 100, 2) if total_rows > 0 else 0
                confidence_records.append({'metric': 'migration_2019_flag_count', 'value': m2019, 'note': f'{pct_m}% — interpret with caution'})
                print(f"  migration_2019: {m2019:,} ({pct_m}%)")

            # Exploratory confidence distribution
            avg_null_pct = np.mean(null_flags) if null_flags else 100.0
            if avg_null_pct <= 10:
                est_confidence = 'ELEVE (estimated)'
            elif avg_null_pct <= 30:
                est_confidence = 'MOYEN (estimated)'
            else:
                est_confidence = 'FAIBLE (estimated)'
            confidence_records.append({'metric': 'estimated_confidence_level', 'value': est_confidence, 'note': f'avg null rate across keys: {avg_null_pct:.1f}%'})
            print(f"  Estimated confidence level: {est_confidence}")
            CONFIDENCE_STATUS = STATUS_READY

        except Exception as e:
            confidence_records.append({'metric': 'data_quality_audit', 'value': 'ERROR', 'note': str(e)})
            print(f"  [ERROR] {e}")

    conf_df = pd.DataFrame(confidence_records)
    out_path = OUTPUT_DIR / 'claim_scoring_confidence_readiness.csv'
    conf_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"\n[EXPORT] Confidence readiness saved: {out_path}")
    display(conf_df)

print(f"\nConfidence readiness status: {CONFIDENCE_STATUS}")

[INFO] Data quality / confidence readiness on: "dwh"."fact_sinistre"
  [ERROR] Execution failed on sql 'SELECT SUM(CASE WHEN "client_sk" IS NULL THEN 1 ELSE 0 END) AS missing_client_count, SUM(CASE WHEN "vehicule_sk" IS NULL THEN 1 ELSE 0 END) AS missing_vehicle_count, SUM(CASE WHEN "contrat_sk" IS NULL THEN 1 ELSE 0 END) AS missing_contract_count, SUM(CASE WHEN "date_survenance_sk" < '2019-01-01' THEN 1 ELSE 0 END) AS migration_2019_count FROM "dwh"."fact_sinistre"': (psycopg2.errors.InvalidTextRepresentation) ERREUR:  syntaxe en entrée invalide pour le type bigint : « 2019-01-01 »
LINE 1: ...tract_count, SUM(CASE WHEN "date_survenance_sk" < '2019-01-0...
                                                             ^

[SQL: SELECT SUM(CASE WHEN "client_sk" IS NULL THEN 1 ELSE 0 END) AS missing_client_count, SUM(CASE WHEN "vehicule_sk" IS NULL THEN 1 ELSE 0 END) AS missing_vehicle_count, SUM(CASE WHEN "contrat_sk" IS NULL THEN 1 ELSE 0 END) AS missing_contract_count, SUM(CASE WHEN "dat

,metric,value,note
0,data_quality_audit,ERROR,Execution failed on sql 'SELECT SUM(CASE WHEN ...



Confidence readiness status: TO_AUDIT


---
## Section 17 — Matrice de disponibilité des signaux

In [16]:
# =============================================================================
# SECTION 17 — SIGNAL READINESS MATRIX
# Consolidates readiness for all signal families and individual features.
# =============================================================================

client_col = AUDIT_STATE.get('client_key_col') or 'NOT_FOUND'
vehicle_col = AUDIT_STATE.get('vehicle_key_col') or 'NOT_FOUND'
contract_col = AUDIT_STATE.get('contract_key_col') or 'NOT_FOUND'
guarantee_col = AUDIT_STATE.get('guarantee_key_col') or 'NOT_FOUND'
geo_col = AUDIT_STATE.get('geo_key_col') or 'NOT_FOUND'
claim_date_col = AUDIT_STATE.get('claim_date_col') or 'NOT_FOUND'
decl_date_col = AUDIT_STATE.get('declaration_date_col') or 'NOT_FOUND'
amount_col = AUDIT_STATE.get('amount_col') or 'NOT_FOUND'

def _ready(col): return col != 'NOT_FOUND' and col is not None

signal_matrix = [
    # ── Client Recurrence ───────────────────────────────────────────────────
    {'signal_family': 'Récurrence client', 'feature_name': 'client_claim_count_total',
     'required_columns': 'client_sk, claim_date', 'detected_columns': f'{client_col}, {claim_date_col}',
     'required_tables': 'fact_claims', 'detected_tables': AUDIT_STATE.get('best_claim_table') or 'NOT_FOUND',
     'readiness_status': CLIENT_RECURRENCE_STATUS,
     'blocking_issue': 'No client key' if not _ready(client_col) else '',
     'recommendation': 'Calculable' if _ready(client_col) and _ready(claim_date_col) else 'Fix client key first',
     'implementation_priority': 'P1'},

    {'signal_family': 'Récurrence client', 'feature_name': 'client_claim_count_12m',
     'required_columns': 'client_sk, claim_date', 'detected_columns': f'{client_col}, {claim_date_col}',
     'required_tables': 'fact_claims', 'detected_tables': AUDIT_STATE.get('best_claim_table') or 'NOT_FOUND',
     'readiness_status': CLIENT_RECURRENCE_STATUS,
     'blocking_issue': 'No client key or date' if not (_ready(client_col) and _ready(claim_date_col)) else '',
     'recommendation': 'Use 365-day rolling window on claim_date',
     'implementation_priority': 'P1'},

    {'signal_family': 'Récurrence client', 'feature_name': 'client_claim_count_24m',
     'required_columns': 'client_sk, claim_date', 'detected_columns': f'{client_col}, {claim_date_col}',
     'required_tables': 'fact_claims', 'detected_tables': AUDIT_STATE.get('best_claim_table') or 'NOT_FOUND',
     'readiness_status': CLIENT_RECURRENCE_STATUS,
     'blocking_issue': 'Sensitive to migration_2019 effect' if _ready(client_col) else 'No client key',
     'recommendation': 'Document migration_2019 impact on 24m window',
     'implementation_priority': 'P1'},

    {'signal_family': 'Récurrence client', 'feature_name': 'days_since_previous_claim',
     'required_columns': 'client_sk, claim_date', 'detected_columns': f'{client_col}, {claim_date_col}',
     'required_tables': 'fact_claims', 'detected_tables': AUDIT_STATE.get('best_claim_table') or 'NOT_FOUND',
     'readiness_status': CLIENT_RECURRENCE_STATUS,
     'blocking_issue': '' if _ready(client_col) and _ready(claim_date_col) else 'Missing key or date',
     'recommendation': 'LAG window function on claim_date per client',
     'implementation_priority': 'P1'},

    # ── Vehicle Recurrence ──────────────────────────────────────────────────
    {'signal_family': 'Récurrence véhicule', 'feature_name': 'vehicle_claim_count_total',
     'required_columns': 'vehicle_sk, claim_date', 'detected_columns': f'{vehicle_col}, {claim_date_col}',
     'required_tables': 'fact_claims', 'detected_tables': AUDIT_STATE.get('best_claim_table') or 'NOT_FOUND',
     'readiness_status': VEHICLE_RECURRENCE_STATUS,
     'blocking_issue': 'No vehicle key' if not _ready(vehicle_col) else '',
     'recommendation': 'Calculable if vehicle_sk available',
     'implementation_priority': 'P1'},

    {'signal_family': 'Récurrence véhicule', 'feature_name': 'vehicle_claim_count_12m',
     'required_columns': 'vehicle_sk, claim_date', 'detected_columns': f'{vehicle_col}, {claim_date_col}',
     'required_tables': 'fact_claims', 'detected_tables': AUDIT_STATE.get('best_claim_table') or 'NOT_FOUND',
     'readiness_status': VEHICLE_RECURRENCE_STATUS,
     'blocking_issue': '' if _ready(vehicle_col) and _ready(claim_date_col) else 'Missing key or date',
     'recommendation': '365-day rolling window on claim_date per vehicle',
     'implementation_priority': 'P1'},

    {'signal_family': 'Récurrence véhicule', 'feature_name': 'days_since_previous_vehicle_claim',
     'required_columns': 'vehicle_sk, claim_date', 'detected_columns': f'{vehicle_col}, {claim_date_col}',
     'required_tables': 'fact_claims', 'detected_tables': AUDIT_STATE.get('best_claim_table') or 'NOT_FOUND',
     'readiness_status': VEHICLE_RECURRENCE_STATUS,
     'blocking_issue': '' if _ready(vehicle_col) and _ready(claim_date_col) else 'Missing key or date',
     'recommendation': 'LAG window function on claim_date per vehicle',
     'implementation_priority': 'P1'},

    # ── VHS ─────────────────────────────────────────────────────────────────
    {'signal_family': 'VHS', 'feature_name': 'linked_vhs_score',
     'required_columns': 'vehicle_sk, vhs_score, inspection_date', 'detected_columns': 'See VHS audit',
     'required_tables': 'fact_vhs_score', 'detected_tables': 'See VHS audit',
     'readiness_status': VHS_STATUS,
     'blocking_issue': 'VHS table not found or linkage too low' if VHS_STATUS != STATUS_PARTIAL else '',
     'recommendation': 'Join by vehicle_sk, filter inspection < claim_date; max 5pts in V1',
     'implementation_priority': 'P2'},

    {'signal_family': 'VHS', 'feature_name': 'vhs_linkage_flag',
     'required_columns': 'vehicle_sk', 'detected_columns': vehicle_col,
     'required_tables': 'fact_vhs_score', 'detected_tables': 'See VHS audit',
     'readiness_status': VHS_STATUS,
     'blocking_issue': '',
     'recommendation': 'Binary flag: 1 if VHS record exists before claim_date',
     'implementation_priority': 'P2'},

    # ── Third-party / Driver ─────────────────────────────────────────────────
    {'signal_family': 'Récurrence tiers', 'feature_name': 'third_party_claim_count_total',
     'required_columns': 'third_party_id', 'detected_columns': 'See TP audit',
     'required_tables': 'fact_third_party', 'detected_tables': 'See TP audit',
     'readiness_status': TP_DRIVER_STATUS,
     'blocking_issue': 'Third-party data incomplete or missing',
     'recommendation': 'P3 — do not force in V1 if data incomplete',
     'implementation_priority': 'P3'},

    {'signal_family': 'Récurrence tiers', 'feature_name': 'client_third_party_pair_count',
     'required_columns': 'client_sk, third_party_id', 'detected_columns': 'See TP audit',
     'required_tables': 'fact_claims, fact_third_party', 'detected_tables': 'See TP audit',
     'readiness_status': TP_DRIVER_STATUS,
     'blocking_issue': 'Requires both client_sk and third_party_id',
     'recommendation': 'P3 — audited separately after TP data confirmed',
     'implementation_priority': 'P3'},

    # ── Chronology ───────────────────────────────────────────────────────────
    {'signal_family': 'Chronologie', 'feature_name': 'days_claim_to_declaration',
     'required_columns': 'claim_date, declaration_date', 'detected_columns': f'{claim_date_col}, {decl_date_col}',
     'required_tables': 'fact_claims', 'detected_tables': AUDIT_STATE.get('best_claim_table') or 'NOT_FOUND',
     'readiness_status': STATUS_READY if (_ready(claim_date_col) and _ready(decl_date_col)) else STATUS_PARTIAL,
     'blocking_issue': '' if _ready(claim_date_col) and _ready(decl_date_col) else 'Missing date columns',
     'recommendation': 'Direct subtraction; check for negative values',
     'implementation_priority': 'P1'},

    {'signal_family': 'Chronologie', 'feature_name': 'days_contract_start_to_claim',
     'required_columns': 'claim_date, contract_start_date', 'detected_columns': f'{claim_date_col}, {contract_col}',
     'required_tables': 'fact_claims, dim_contract', 'detected_tables': 'Requires join audit',
     'readiness_status': CHRONOLOGY_STATUS,
     'blocking_issue': 'Requires contract join — audit contract start date availability',
     'recommendation': 'Confirm contract start date joinable to claim',
     'implementation_priority': 'P1'},

    {'signal_family': 'Chronologie', 'feature_name': 'claim_before_contract_start_flag',
     'required_columns': 'claim_date, contract_start_date', 'detected_columns': f'{claim_date_col}, {contract_col}',
     'required_tables': 'fact_claims, dim_contract', 'detected_tables': 'Requires join audit',
     'readiness_status': CHRONOLOGY_STATUS,
     'blocking_issue': 'Derived from days_contract_start_to_claim',
     'recommendation': 'Boolean: claim_date < contract_start_date',
     'implementation_priority': 'P1'},

    {'signal_family': 'Chronologie', 'feature_name': 'recent_contract_change_flag',
     'required_columns': 'contract modification history', 'detected_columns': 'TO_AUDIT',
     'required_tables': 'dim_contract history', 'detected_tables': 'TO_AUDIT',
     'readiness_status': STATUS_TO_AUDIT,
     'blocking_issue': 'Requires modification history availability confirmation',
     'recommendation': 'Audit contract modification history table',
     'implementation_priority': 'P1'},

    # ── Amount ───────────────────────────────────────────────────────────────
    {'signal_family': 'Montant', 'feature_name': 'claim_amount',
     'required_columns': 'montant / amount', 'detected_columns': amount_col,
     'required_tables': 'fact_claims', 'detected_tables': AUDIT_STATE.get('best_claim_table') or 'NOT_FOUND',
     'readiness_status': STATUS_READY if _ready(amount_col) else STATUS_NOT_AVAILABLE,
     'blocking_issue': '' if _ready(amount_col) else 'No amount column found',
     'recommendation': 'Validate non-null coverage and check for negatives',
     'implementation_priority': 'P1'},

    {'signal_family': 'Montant', 'feature_name': 'amount_vs_guarantee_median_ratio',
     'required_columns': 'amount, guarantee_key', 'detected_columns': f'{amount_col}, {guarantee_col}',
     'required_tables': 'fact_claims, dim_guarantee', 'detected_tables': 'fact_claims',
     'readiness_status': STATUS_PARTIAL if _ready(amount_col) and _ready(guarantee_col) else STATUS_TO_AUDIT,
     'blocking_issue': '' if _ready(amount_col) and _ready(guarantee_col) else 'Guarantee or amount column missing',
     'recommendation': 'Requires sufficient claim count per guarantee group',
     'implementation_priority': 'P1'},

    {'signal_family': 'Montant', 'feature_name': 'high_amount_flag',
     'required_columns': 'amount, guarantee_key', 'detected_columns': f'{amount_col}, {guarantee_col}',
     'required_tables': 'fact_claims', 'detected_tables': AUDIT_STATE.get('best_claim_table') or 'NOT_FOUND',
     'readiness_status': STATUS_PARTIAL if _ready(amount_col) else STATUS_NOT_AVAILABLE,
     'blocking_issue': '',
     'recommendation': 'Derived from amount_percentile_by_guarantee (>=90)',
     'implementation_priority': 'P1'},

    # ── GEO ──────────────────────────────────────────────────────────────────
    {'signal_family': 'Cohérence géographique', 'feature_name': 'geo_mismatch_flag',
     'required_columns': 'claim_geo_sk, client_geo_sk, agency_geo_sk', 'detected_columns': geo_col,
     'required_tables': 'fact_claims, dim_geo, dim_client, dim_agency', 'detected_tables': 'Requires multi-table join',
     'readiness_status': GEO_STATUS,
     'blocking_issue': 'ETL GEO must be stabilized before activation',
     'recommendation': 'Activate only after GEO ETL confirmed stable',
     'implementation_priority': 'P2'},

    {'signal_family': 'Cohérence géographique', 'feature_name': 'unknown_geo_flag',
     'required_columns': 'claim_geo_sk', 'detected_columns': geo_col,
     'required_tables': 'fact_claims, dim_geo', 'detected_tables': 'fact_claims',
     'readiness_status': GEO_STATUS,
     'blocking_issue': 'UNKNOWN geo = data quality, not suspicion',
     'recommendation': 'Use to set confidence_level, not attention score',
     'implementation_priority': 'P2'},

    # ── Data quality / Confidence ────────────────────────────────────────────
    {'signal_family': 'Qualité des données', 'feature_name': 'missing_keys_count',
     'required_columns': 'client_sk, vehicle_sk, contract_sk, agency_sk', 'detected_columns': f'{client_col}, {vehicle_col}, {contract_col}',
     'required_tables': 'fact_claims', 'detected_tables': AUDIT_STATE.get('best_claim_table') or 'NOT_FOUND',
     'readiness_status': CONFIDENCE_STATUS,
     'blocking_issue': '',
     'recommendation': 'Sum of IS NULL indicators — always computable',
     'implementation_priority': 'P1'},

    {'signal_family': 'Qualité des données', 'feature_name': 'confidence_level',
     'required_columns': 'missing_keys_count, unknown_dimensions_count, weak_join_flag',
     'detected_columns': 'Derived',
     'required_tables': 'fact_claims', 'detected_tables': AUDIT_STATE.get('best_claim_table') or 'NOT_FOUND',
     'readiness_status': CONFIDENCE_STATUS,
     'blocking_issue': '',
     'recommendation': 'Categorical: ELEVE / MOYEN / FAIBLE — always computable',
     'implementation_priority': 'P1'},

    {'signal_family': 'Qualité des données', 'feature_name': 'migration_2019_flag',
     'required_columns': 'claim_date', 'detected_columns': claim_date_col,
     'required_tables': 'fact_claims', 'detected_tables': AUDIT_STATE.get('best_claim_table') or 'NOT_FOUND',
     'readiness_status': STATUS_READY if _ready(claim_date_col) else STATUS_NOT_AVAILABLE,
     'blocking_issue': '',
     'recommendation': 'claim_date < 2019-01-01 — verify exact migration cutoff date',
     'implementation_priority': 'P1'},
]

signal_matrix_df = pd.DataFrame(signal_matrix)

out_path = OUTPUT_DIR / 'claim_scoring_signal_readiness_matrix.csv'
signal_matrix_df.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"[EXPORT] Signal readiness matrix saved: {out_path}")

# Summary by priority and status
print("\nSignal readiness summary by priority:")
print(signal_matrix_df.groupby(['implementation_priority', 'readiness_status']).size().reset_index(name='count').to_string(index=False))
display(signal_matrix_df[['signal_family', 'feature_name', 'readiness_status', 'implementation_priority', 'recommendation']])

[EXPORT] Signal readiness matrix saved: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\data_readiness\claim_scoring_signal_readiness_matrix.csv

Signal readiness summary by priority:
implementation_priority readiness_status  count
                     P1          PARTIAL      4
                     P1            READY     10
                     P1         TO_AUDIT      3
                     P2        NOT_READY      4
                     P3          PARTIAL      2


,signal_family,feature_name,readiness_status,implementation_priority,recommendation
0,Récurrence client,client_claim_count_total,READY,P1,Calculable
1,Récurrence client,client_claim_count_12m,READY,P1,Use 365-day rolling window on claim_date
2,Récurrence client,client_claim_count_24m,READY,P1,Document migration_2019 impact on 24m window
3,Récurrence client,days_since_previous_claim,READY,P1,LAG window function on claim_date per client
4,Récurrence véhicule,vehicle_claim_count_total,READY,P1,Calculable if vehicle_sk available
5,Récurrence véhicule,vehicle_claim_count_12m,READY,P1,365-day rolling window on claim_date per vehicle
6,Récurrence véhicule,days_since_previous_vehicle_claim,READY,P1,LAG window function on claim_date per vehicle
7,VHS,linked_vhs_score,NOT_READY,P2,"Join by vehicle_sk, filter inspection < claim_..."
8,VHS,vhs_linkage_flag,NOT_READY,P2,Binary flag: 1 if VHS record exists before cla...
9,Récurrence tiers,third_party_claim_count_total,PARTIAL,P3,P3 — do not force in V1 if data incomplete


---
## Section 18 — Export des rapports de qualité locaux

In [17]:
# =============================================================================
# SECTION 18 — EXPORT LOCAL QUALITY REPORTS
# Generate a Markdown summary report of the entire audit.
# =============================================================================

import textwrap

def _matrix_summary_lines(df):
    """Build a compact Markdown table from the signal readiness matrix."""
    lines = ['| Famille | Feature | Statut | Priorité |',
             '|---|---|---|---|']
    for _, r in df.iterrows():
        lines.append(f"| {r['signal_family']} | {r['feature_name']} | {r['readiness_status']} | {r['implementation_priority']} |")
    return '\n'.join(lines)

md_summary = textwrap.dedent(f"""
# Rapport de préparation des données — Score d'attention dossier IRIS

> Généré automatiquement par le notebook `01_claim_scoring_data_readiness.ipynb`

## Métadonnées d'exécution

| Paramètre | Valeur |
|---|---|
| Date d'exécution | {RUN_TIMESTAMP} |
| Statut connexion DB | {DB_CONNECTION_STATUS} |
| Note connexion | {DB_CONNECTION_NOTE} |
| Table sinistres candidate | {AUDIT_STATE.get('best_claim_schema', 'N/A')}.{AUDIT_STATE.get('best_claim_table', 'N/A')} |
| Total lignes sinistres | {AUDIT_STATE.get('total_claim_rows', 'N/A')} |

## Colonnes clés détectées

| Clé | Colonne détectée |
|---|---|
| client_key | {AUDIT_STATE.get('client_key_col', 'NOT_FOUND')} |
| vehicle_key | {AUDIT_STATE.get('vehicle_key_col', 'NOT_FOUND')} |
| contract_key | {AUDIT_STATE.get('contract_key_col', 'NOT_FOUND')} |
| guarantee_key | {AUDIT_STATE.get('guarantee_key_col', 'NOT_FOUND')} |
| agency_key | {AUDIT_STATE.get('agency_key_col', 'NOT_FOUND')} |
| geo_key | {AUDIT_STATE.get('geo_key_col', 'NOT_FOUND')} |
| claim_date | {AUDIT_STATE.get('claim_date_col', 'NOT_FOUND')} |
| declaration_date | {AUDIT_STATE.get('declaration_date_col', 'NOT_FOUND')} |
| amount | {AUDIT_STATE.get('amount_col', 'NOT_FOUND')} |

## Statuts de disponibilité par famille de signaux

| Famille | Statut | Priorité |
|---|---|---|
| Récurrence client | {CLIENT_RECURRENCE_STATUS} | P1 |
| Récurrence véhicule | {VEHICLE_RECURRENCE_STATUS} | P1 |
| Chronologie | {CHRONOLOGY_STATUS} | P1 |
| Montant | READY si amount_col détecté | P1 |
| Qualité données / Confiance | {CONFIDENCE_STATUS} | P1 |
| Cohérence géographique | {GEO_STATUS} | P2 |
| VHS | {VHS_STATUS} | P2 |
| Récurrence tiers / conducteur | {TP_DRIVER_STATUS} | P3 |

## Matrice de disponibilité des features

{_matrix_summary_lines(signal_matrix_df) if not signal_matrix_df.empty else 'Non disponible'}

## Rapports CSV exportés

| Rapport | Fichier |
|---|---|
| Inventaire des tables | `claim_scoring_table_inventory.csv` |
| Candidats table sinistres | `claim_scoring_claim_table_candidates.csv` |
| Couverture des clés | `claim_scoring_key_coverage.csv` |
| Qualité des dates | `claim_scoring_date_quality.csv` |
| Qualité des montants | `claim_scoring_amount_quality.csv` |
| Récurrence client | `claim_scoring_client_recurrence_readiness.csv` |
| Récurrence véhicule | `claim_scoring_vehicle_recurrence_readiness.csv` |
| Tiers / conducteur | `claim_scoring_third_party_driver_readiness.csv` |
| Chronologie | `claim_scoring_chronology_readiness.csv` |
| GEO | `claim_scoring_geo_readiness.csv` |
| VHS | `claim_scoring_vhs_linkage_readiness.csv` |
| Confiance | `claim_scoring_confidence_readiness.csv` |
| Matrice signaux | `claim_scoring_signal_readiness_matrix.csv` |

## Prochaine étape recommandée

Valider les familles P1 (récurrence client, récurrence véhicule, chronologie, montant, qualité des données) avant toute implémentation.

**Le passage à l'implémentation du score V1 ne doit être effectué qu'après validation des familles P1 et documentation des limites P2/P3.**

---
_Ce rapport est généré automatiquement. Aucun score n'a été calculé et aucune écriture en base n'a été effectuée._
""").strip()

summary_path = OUTPUT_DIR / 'claim_scoring_data_readiness_summary.md'
summary_path.write_text(md_summary, encoding='utf-8')
print(f"[EXPORT] Markdown summary saved: {summary_path}")
print(f"\nAll reports exported to: {OUTPUT_DIR}")
print(f"\nExported files:")
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    print(f"  {f.name}")
for f in sorted(OUTPUT_DIR.glob('*.md')):
    print(f"  {f.name}")

[EXPORT] Markdown summary saved: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\data_readiness\claim_scoring_data_readiness_summary.md

All reports exported to: C:\Users\wiem\Downloads\Projet PFE\IRIS_AUTO_FRAUD\data\quality_reports\scoring\data_readiness

Exported files:
  claim_scoring_amount_by_guarantee.csv
  claim_scoring_amount_quality.csv
  claim_scoring_chronology_readiness.csv
  claim_scoring_claim_table_candidates.csv
  claim_scoring_client_recurrence_readiness.csv
  claim_scoring_confidence_readiness.csv
  claim_scoring_date_quality.csv
  claim_scoring_geo_readiness.csv
  claim_scoring_key_coverage.csv
  claim_scoring_signal_readiness_matrix.csv
  claim_scoring_table_inventory.csv
  claim_scoring_third_party_driver_readiness.csv
  claim_scoring_vehicle_recurrence_readiness.csv
  claim_scoring_vhs_linkage_readiness.csv
  claim_scoring_data_readiness_summary.md


---
## Section 19 — Synthèse exécutive

Tableau de synthèse de la préparation des données pour le score d'attention dossier IRIS.

In [18]:
# =============================================================================
# SECTION 19 — EXECUTIVE SUMMARY TABLE
# =============================================================================

amount_col_found = AUDIT_STATE.get('amount_col')
amount_status = STATUS_READY if amount_col_found else STATUS_NOT_AVAILABLE

exec_summary = pd.DataFrame([
    {
        'Domaine': 'Table centrale sinistres',
        'Statut': STATUS_READY if AUDIT_STATE.get('best_claim_table') else STATUS_NOT_FOUND,
        'Constat principal': f"{AUDIT_STATE.get('best_claim_schema','?')}.{AUDIT_STATE.get('best_claim_table','?')} — {AUDIT_STATE.get('total_claim_rows','?'):,} lignes" if AUDIT_STATE.get('best_claim_table') else 'Aucune table détectée',
        'Action recommandée': 'Confirmer grain et déduplication' if AUDIT_STATE.get('best_claim_table') else 'Identifier la table source sinistres',
    },
    {
        'Domaine': 'Récurrence client (P1)',
        'Statut': CLIENT_RECURRENCE_STATUS,
        'Constat principal': f'client_key={AUDIT_STATE.get("client_key_col","NOT_FOUND")}',
        'Action recommandée': 'Implémenter la feature de récurrence client si READY ou PARTIAL',
    },
    {
        'Domaine': 'Récurrence véhicule (P1)',
        'Statut': VEHICLE_RECURRENCE_STATUS,
        'Constat principal': f'vehicle_key={AUDIT_STATE.get("vehicle_key_col","NOT_FOUND")}',
        'Action recommandée': 'Implémenter la feature de récurrence véhicule si READY ou PARTIAL',
    },
    {
        'Domaine': 'Tiers / conducteur (P3)',
        'Statut': TP_DRIVER_STATUS,
        'Constat principal': 'Données tiers probablement incomplètes en V1',
        'Action recommandée': 'Reporter en P3 — ne pas bloquer V1 sur ce point',
    },
    {
        'Domaine': 'Chronologie (P1)',
        'Statut': CHRONOLOGY_STATUS,
        'Constat principal': f'claim_date={AUDIT_STATE.get("claim_date_col","?")} / decl_date={AUDIT_STATE.get("declaration_date_col","?")}',
        'Action recommandée': 'Confirmer disponibilité contract start date via jointure',
    },
    {
        'Domaine': 'Montant (P1)',
        'Statut': amount_status,
        'Constat principal': f'amount_col={amount_col_found or "NOT_FOUND"}',
        'Action recommandée': 'Valider distribution et couverture non-null',
    },
    {
        'Domaine': 'Cohérence GEO (P2)',
        'Statut': GEO_STATUS,
        'Constat principal': f'geo_col={AUDIT_STATE.get("geo_key_col","NOT_FOUND")} — ETL GEO doit être stabilisé',
        'Action recommandée': 'Activer uniquement après confirmation ETL GEO stable',
    },
    {
        'Domaine': 'VHS (P2)',
        'Statut': VHS_STATUS,
        'Constat principal': 'VHS = signal technique limité (max 5pts/100)',
        'Action recommandée': 'Vérifier taux de liaison VHS/véhicule — absence ne bloque pas V1',
    },
    {
        'Domaine': 'Qualité données / Confiance (P1)',
        'Statut': CONFIDENCE_STATUS,
        'Constat principal': 'Clés manquantes → confidence, pas attention',
        'Action recommandée': 'Toujours calculable — implémenter en P1',
    },
])

STATUS_EMOJI = {
    STATUS_READY: '✅ READY',
    STATUS_PARTIAL: '⚠️ PARTIAL',
    STATUS_NOT_READY: '❌ NOT_READY',
    STATUS_NOT_AVAILABLE: '⛔ NOT_AVAILABLE',
    STATUS_TO_AUDIT: '🔍 TO_AUDIT',
}

exec_summary['Statut'] = exec_summary['Statut'].map(lambda s: STATUS_EMOJI.get(s, s))

print("\n" + "="*80)
print("EXECUTIVE SUMMARY — Audit de préparation des données")
print("="*80)
display(exec_summary)


EXECUTIVE SUMMARY — Audit de préparation des données


,Domaine,Statut,Constat principal,Action recommandée
0,Table centrale sinistres,✅ READY,"dwh.fact_sinistre — 381,893 lignes",Confirmer grain et déduplication
1,Récurrence client (P1),✅ READY,client_key=client_sk,Implémenter la feature de récurrence client si...
2,Récurrence véhicule (P1),✅ READY,vehicle_key=vehicule_sk,Implémenter la feature de récurrence véhicule ...
3,Tiers / conducteur (P3),⚠️ PARTIAL,Données tiers probablement incomplètes en V1,Reporter en P3 — ne pas bloquer V1 sur ce point
4,Chronologie (P1),⚠️ PARTIAL,claim_date=date_survenance_sk / decl_date=date...,Confirmer disponibilité contract start date vi...
5,Montant (P1),✅ READY,amount_col=montant_evaluation,Valider distribution et couverture non-null
6,Cohérence GEO (P2),❌ NOT_READY,geo_col=None — ETL GEO doit être stabilisé,Activer uniquement après confirmation ETL GEO ...
7,VHS (P2),❌ NOT_READY,VHS = signal technique limité (max 5pts/100),Vérifier taux de liaison VHS/véhicule — absenc...
8,Qualité données / Confiance (P1),🔍 TO_AUDIT,"Clés manquantes → confidence, pas attention",Toujours calculable — implémenter en P1


---
## Section 20 — Recommandation pour l'étape suivante

### Logique de recommandation

- **Si les familles P1 sont READY ou PARTIAL** → Passer à l'implémentation prototype des features (étape 3 de la feuille de route).
- **Si des familles P1 sont NOT_READY** → Corriger les jointures ou données DWH avant de continuer.
- **P2 et P3** → Ne pas bloquer la V1 sur ces familles.

In [19]:
# =============================================================================
# SECTION 20 — NEXT-STEP RECOMMENDATION
# =============================================================================

P1_STATUSES = {
    'Récurrence client': CLIENT_RECURRENCE_STATUS,
    'Récurrence véhicule': VEHICLE_RECURRENCE_STATUS,
    'Chronologie': CHRONOLOGY_STATUS,
    'Montant': STATUS_READY if AUDIT_STATE.get('amount_col') else STATUS_NOT_AVAILABLE,
    'Qualité données': CONFIDENCE_STATUS,
}

p1_ok = sum(1 for s in P1_STATUSES.values() if s in (STATUS_READY, STATUS_PARTIAL))
p1_total = len(P1_STATUSES)
p1_blocked = [name for name, s in P1_STATUSES.items() if s in (STATUS_NOT_READY, STATUS_NOT_AVAILABLE)]

print("\n" + "="*80)
print("RECOMMANDATION — Étape suivante")
print("="*80)
print(f"\nFamilles P1 prêtes ou partiellement prêtes : {p1_ok}/{p1_total}")

for name, status in P1_STATUSES.items():
    emoji = STATUS_EMOJI.get(status, status)
    print(f"  {emoji}  {name}")

print()

if p1_blocked:
    print("⚠️  Les familles P1 suivantes sont bloquantes :")
    for b in p1_blocked:
        print(f"   - {b}: {P1_STATUSES[b]}")
    print()
    print("➡️  RECOMMANDATION : Corriger les données DWH ou les jointures manquantes avant d'implémenter les features.")
    NEXT_STEP = 'FIX_DWH_JOINS'
else:
    print(f"✅ {p1_ok}/{p1_total} familles P1 sont READY ou PARTIAL.")
    print()
    print("➡️  RECOMMANDATION : Passer à l'implémentation du prototype de features (Étape 3 de la feuille de route).")
    print("   Prochains fichiers à créer :")
    print("   - notebooks/validation_scoring/02_claim_feature_prototype.ipynb (exploration SQL read-only)")
    print("   - docs/scoring/claim_attention_score_rules_v1.md (règles de scoring V1)")
    NEXT_STEP = 'IMPLEMENT_FEATURES'

print()
print("P2 — À activer ultérieurement :")
print(f"  GEO          : {GEO_STATUS} — dépend de la stabilisation ETL GEO")
print(f"  VHS          : {VHS_STATUS} — signal technique limité, absence ne bloque pas")
print()
print("P3 — Reporter après confirmation disponibilité données tiers :")
print(f"  Tiers/driver : {TP_DRIVER_STATUS}")
print()

# Final audit compliance check
print("="*80)
print("CONTRÔLES DE CONFORMITÉ DE L'AUDIT")
print("="*80)
compliance = [
    ('Notebook créé', 'OUI'),
    ('Chemin', 'notebooks/validation_scoring/01_claim_scoring_data_readiness.ipynb'),
    ('Dossier de sortie créé', 'OUI' if OUTPUT_DIR.exists() else 'NON'),
    ('Garde-fou lecture seule inclus', 'OUI'),
    ('Rapports planifiés/exportés', 'OUI'),
    ('Code de production modifié', 'NON'),
    ('Écriture en base de données effectuée', 'NON'),
    ('Module VHS modifié', 'NON'),
    ('Score implémenté', 'NON'),
    ('Machine Learning utilisé', 'NON'),
    ('SHAP utilisé', 'NON'),
    ('XGBoost utilisé', 'NON'),
]
for check, value in compliance:
    icon = '✅' if value in ('OUI', 'NON') else '⚠️'
    print(f"  {icon} {check} : {value}")

print()
print("✅ Read-only claim scoring data readiness notebook created successfully.")
print("   No production code was modified, no VHS logic was changed,")
print("   no scoring was implemented, and no database write was performed.")


RECOMMANDATION — Étape suivante

Familles P1 prêtes ou partiellement prêtes : 4/5
  ✅ READY  Récurrence client
  ✅ READY  Récurrence véhicule
  ⚠️ PARTIAL  Chronologie
  ✅ READY  Montant
  🔍 TO_AUDIT  Qualité données

✅ 4/5 familles P1 sont READY ou PARTIAL.

➡️  RECOMMANDATION : Passer à l'implémentation du prototype de features (Étape 3 de la feuille de route).
   Prochains fichiers à créer :
   - notebooks/validation_scoring/02_claim_feature_prototype.ipynb (exploration SQL read-only)
   - docs/scoring/claim_attention_score_rules_v1.md (règles de scoring V1)

P2 — À activer ultérieurement :
  GEO          : NOT_READY — dépend de la stabilisation ETL GEO
  VHS          : NOT_READY — signal technique limité, absence ne bloque pas

P3 — Reporter après confirmation disponibilité données tiers :
  Tiers/driver : PARTIAL

CONTRÔLES DE CONFORMITÉ DE L'AUDIT
  ✅ Notebook créé : OUI
  ⚠️ Chemin : notebooks/validation_scoring/01_claim_scoring_data_readiness.ipynb
  ✅ Dossier de sortie créé :

---

> **Le passage à l'implémentation du score V1 ne doit être effectué qu'après validation des familles P1 et documentation des limites P2/P3.**

---
_Ce notebook est en lecture seule. Aucun score d'attention n'a été calculé. Aucune écriture en base de données n'a été effectuée._  
_IRIS Auto Fraud Decision Platform — PFE BNA Assurances_